# Single Precision in GPU Molecular Dynamics: What Breaks and What Doesn't

Part 2 of the CUDA molecular dynamics project. Part 1 built a 1D Lennard-Jones simulator in double precision. The T4's FP32 throughput is 32 times its FP64 throughput, so the question here is how much of the simulation can run in single precision, and what that does to energy conservation and to run time.

The FP64 reference is `md_differential.cu` from part 1: shifted-force Lennard-Jones, cutoff 2.5, N = 65,536, dt = 1e-4. It reaches a relative energy error of about 3e-8 over 30,000 steps (t = 3).

## Contents

1. Everything in FP32
2. Mixed precision
3. Kernel benchmark
4. Where the time goes
5. Same source, FP64 vs mixed, three seeds
6. Keeping the state on the GPU
7. Validation against OpenMM
8. Summary

## 1. Everything in FP32

All state (positions, velocities, accelerations, forces) and the energy diagnostics are `float`, with `f`-suffixed literals so nothing silently promotes back to double. This version runs only two steps, with diagnostics added to look at what happens to the positions. (An earlier run of this program with `STEPS = 3000` gave a relative energy error of 15% at t = 0.3, with E - E0 growing like step^2; that run is not repeated here, set `STEPS` to reproduce it.)

In [1]:
%%writefile md_fp32.cu
#include <iostream>
#include <vector>
#include <random>
#include <cmath>
#include <iomanip>
#include <algorithm>
#include <cuda_runtime.h>

#define TILE_SIZE 256

const float SIGMA = 1.0f;
const float EPSILON = 1.0f;
const float MASS = 1.0f;
const float RC = 2.5f;
const float LCELL = 2.5f;

const float DT = 0.0001f;
const int N = 65536;
const int STEPS = 2;
const int CHECKPOINT = 100;

float ljForce(float r) {
    float invR = 1.0f / r;
    float sr = SIGMA * invR;

    float sr6 =
        sr * sr * sr * sr * sr * sr;

    float sr12 = sr6 * sr6;

    return 24.0f * EPSILON * invR *
           (2.0f * sr12 - sr6);
}

float ljPotential(float r) {
    float invR = 1.0f / r;
    float sr = SIGMA * invR;

    float sr6 =
        sr * sr * sr * sr * sr * sr;

    float sr12 = sr6 * sr6;

    return 4.0f * EPSILON *
           (sr12 - sr6);
}

const float FC = ljForce(RC);

__global__ void cellListForceKernel(
    const float *sortedR,
    const int *cellStart,
    float *F_sorted,
    int N,
    int numCells,
    float r_min,
    float FC
) {
    int k =
        blockIdx.x * blockDim.x +
        threadIdx.x;

    if (k >= N) {
        return;
    }

    float myR = sortedR[k];

    int myCell =
        (int)floor(
            (myR - r_min) / LCELL
        );

    if (myCell < 0) {
        myCell = 0;
    }

    if (myCell >= numCells) {
        myCell = numCells - 1;
    }

    float Fi = 0.0f;

    for (int c = myCell - 1;
         c <= myCell + 1;
         c++) {

        if (c < 0 || c >= numCells) {
            continue;
        }

        for (int m = cellStart[c];
             m < cellStart[c + 1];
             m++) {

            if (m == k) {
                continue;
            }

            float dx =
                myR - sortedR[m];

            float dist = fabs(dx);

            if (dist < RC) {
                float invR = 1.0f / dist;

                float sr =
                    SIGMA * invR;

                float sr6 =
                    sr * sr * sr * sr * sr * sr;

                float sr12 =
                    sr6 * sr6;

                float force =
                    24.0f * EPSILON * invR *
                    (2.0f * sr12 - sr6) - FC;

                Fi +=
                    force * (dx * invR);
            }
        }
    }

    F_sorted[k] = Fi;
}

void buildCellList(
    const std::vector<float>& r,
    std::vector<float>& sortedR,
    std::vector<int>& sortedIndex,
    std::vector<int>& cellStart,
    float& r_min,
    float& r_max,
    int& numCells
) {
    r_min = r[0];
    r_max = r[0];

    for (int i = 1; i < N; i++) {
        r_min =
            std::min(r_min, r[i]);

        r_max =
            std::max(r_max, r[i]);
    }

    numCells =
        (int)ceil(
            (r_max - r_min) / LCELL
        );

    if (numCells < 1) {
        numCells = 1;
    }

    std::vector<int> cellCount(
        numCells,
        0
    );

    for (int i = 0; i < N; i++) {
        int cell =
            (int)floor(
                (r[i] - r_min) / LCELL
            );

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        cellCount[cell]++;
    }

    cellStart.assign(
        numCells + 1,
        0
    );

    for (int c = 0;
         c < numCells;
         c++) {

        cellStart[c + 1] =
            cellStart[c] +
            cellCount[c];
    }

    std::vector<int> cursor =
        cellStart;

    sortedR.resize(N);
    sortedIndex.resize(N);

    for (int i = 0; i < N; i++) {
        int cell =
            (int)floor(
                (r[i] - r_min) / LCELL
            );

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        int slot =
            cursor[cell];

        sortedR[slot] =
            r[i];

        sortedIndex[slot] =
            i;

        cursor[cell]++;
    }
}

void computeForcesGPU(
    const std::vector<float>& sortedR,
    const std::vector<int>& cellStart,
    const std::vector<int>& sortedIndex,
    std::vector<float>& F,
    float r_min,
    int numCells,
    float* d_sortedR,
    int* d_cellStart,
    float* d_F_sorted,
    float FC
) {
    cudaMemcpy(
        d_sortedR,
        sortedR.data(),
        N * sizeof(float),
        cudaMemcpyHostToDevice
    );

    cudaMemcpy(
        d_cellStart,
        cellStart.data(),
        (numCells + 1) * sizeof(int),
        cudaMemcpyHostToDevice
    );

    int blocks =
        (N + TILE_SIZE - 1) /
        TILE_SIZE;

    cellListForceKernel<<<
        blocks,
        TILE_SIZE
    >>>(
        d_sortedR,
        d_cellStart,
        d_F_sorted,
        N,
        numCells,
        r_min,
        FC
    );

    cudaError_t error =
        cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr
            << "Kernel launch error: "
            << cudaGetErrorString(error)
            << '\n';

        exit(1);
    }

    std::vector<float> F_sorted(N);

    cudaMemcpy(
        F_sorted.data(),
        d_F_sorted,
        N * sizeof(float),
        cudaMemcpyDeviceToHost
    );

    for (int k = 0; k < N; k++) {
        int originalIndex =
            sortedIndex[k];

        F[originalIndex] =
            F_sorted[k];
    }
}

float computeKE(
    const std::vector<float>& v
) {
    float KE = 0.0f;

    for (int i = 0; i < N; i++) {
        KE +=
            0.5f *
            MASS *
            v[i] *
            v[i];
    }

    return KE;
}

float computePE(
    const std::vector<float>& r,
    const std::vector<int>& sortedIndex,
    const std::vector<int>& cellStart,
    float r_min,
    int numCells,
    float FC
) {
    float PE = 0.0f;

    for (int i = 0; i < N; i++) {

        int myCell =
            (int)floor(
                (r[i] - r_min) /
                LCELL
            );

        if (myCell < 0) {
            myCell = 0;
        }

        if (myCell >= numCells) {
            myCell = numCells - 1;
        }

        for (int c = myCell - 1;
             c <= myCell + 1;
             c++) {

            if (c < 0 ||
                c >= numCells) {
                continue;
            }

            for (int m = cellStart[c];
                 m < cellStart[c + 1];
                 m++) {

                int j =
                    sortedIndex[m];

                if (j <= i) {
                    continue;
                }

                float dx =
                    r[i] - r[j];

                float dist =
                    fabs(dx);

                if (dist < RC) {
                    PE +=
                        ljPotential(dist) - ljPotential(RC)
                        + FC * (dist - RC);
                }
            }
        }
    }

    return PE;
}

float minimumGap(
    const std::vector<float>& sortedR
) {
    float minDist = 1e30f;

    for (int i = 1; i < N; i++) {
        float gap =
            sortedR[i] -
            sortedR[i - 1];

        if (gap < minDist) {
            minDist = gap;
        }
    }

    return minDist;
}

int main() {

    std::vector<float> r(N);
    std::vector<float> v(N);
    std::vector<float> a(N);
    std::vector<float> a_new(N);
    std::vector<float> F(N);

    float spacing = 1.5f;
    float jitterAmplitude = 0.2f;

    std::mt19937 gen(42);

    std::uniform_real_distribution<float> jitter(
        -jitterAmplitude,
        jitterAmplitude
    );

    for (int i = 0; i < N; i++) {
        r[i] =
            i * spacing +
            jitter(gen);
    }

    for (int i = 0; i < N; i++) {
        if (i % 2 == 0) {
            v[i] = 0.01f;
        } else {
            v[i] = -0.01f;
        }
    }

    std::vector<float> sortedR;
    std::vector<int> sortedIndex;
    std::vector<int> cellStart;

    float r_min;
    float r_max;
    int numCells;

    buildCellList(
        r,
        sortedR,
        sortedIndex,
        cellStart,
        r_min,
        r_max,
        numCells
    );

    float* d_sortedR;
    int* d_cellStart;
    float* d_F_sorted;

    cudaMalloc(
        &d_sortedR,
        N * sizeof(float)
    );

    cudaMalloc(
        &d_cellStart,
        (numCells + 1) *
        sizeof(int)
    );

    cudaMalloc(
        &d_F_sorted,
        N * sizeof(float)
    );

    computeForcesGPU(
        sortedR,
        cellStart,
        sortedIndex,
        F,
        r_min,
        numCells,
        d_sortedR,
        d_cellStart,
        d_F_sorted,
        FC
    );

    for (int i = 0; i < N; i++) {
        a[i] =
            F[i] / MASS;
    }

    double sumA2 = 0.0;
    for (int i = 0; i < N; i++) {
        sumA2 += (double)a[i] * (double)a[i];
    }
    std::cout << "DIAG sum(a^2) at t=0 = " << sumA2
              << "  (mean a^2 = " << sumA2 / N << ")\n";

    float KE0 =
        computeKE(v);

    float PE0 =
        computePE(
            r,
            sortedIndex,
            cellStart,
            r_min,
            numCells,
            FC
        );

    float E0 =
        KE0 + PE0;

    std::cout
        << std::scientific
        << std::setprecision(9);

    std::cout
        << "Precision = FLOAT (FP32)\n";

    std::cout
        << "N = "
        << N
        << '\n';

    std::cout
        << "dt = "
        << DT
        << '\n';

    std::cout
        << "Total simulated time = "
        << DT * STEPS
        << '\n';

    std::cout
        << "Initial KE = "
        << KE0
        << '\n';

    std::cout
        << "Initial PE = "
        << PE0
        << '\n';

    std::cout
        << "Initial E  = "
        << E0
        << '\n';

    std::cout << '\n';

    std::cout
        << std::setw(10)
        << "Step"
        << std::setw(18)
        << "Time"
        << std::setw(18)
        << "MinGap"
        << std::setw(18)
        << "KE"
        << std::setw(18)
        << "PE"
        << std::setw(18)
        << "Total E"
        << std::setw(18)
        << "E-E0"
        << '\n';

    std::cout
        << std::setw(10)
        << 0
        << std::setw(18)
        << 0.0f
        << std::setw(18)
        << minimumGap(sortedR)
        << std::setw(18)
        << KE0
        << std::setw(18)
        << PE0
        << std::setw(18)
        << E0
        << std::setw(18)
        << 0.0f
        << '\n';

    for (int step = 1;
         step <= STEPS;
         step++) {

        std::vector<float> r_before;
        if (step == 1) {
            r_before = r;
        }

        for (int i = 0; i < N; i++) {
            r[i] +=
                v[i] * DT +
                0.5f *
                a[i] *
                DT *
                DT;
        }

        if (step == 1) {
            int unchanged = 0;
            for (int i = 0; i < N; i++) {
                if (r[i] == r_before[i]) {
                    unchanged++;
                }
            }
            float rl = r_before[N - 1];
            std::cout << "DIAG step 1: unchanged positions = " << unchanged
                      << " / " << N << '\n';
            std::cout << "DIAG r[N-1] = " << rl
                      << ", ULP there = " << nextafterf(rl, INFINITY) - rl
                      << ", requested move v*dt+0.5*a*dt^2 = "
                      << v[N - 1] * DT + 0.5f * a[N - 1] * DT * DT << '\n';
        }

        buildCellList(
            r,
            sortedR,
            sortedIndex,
            cellStart,
            r_min,
            r_max,
            numCells
        );

        cudaFree(d_cellStart);

        cudaMalloc(
            &d_cellStart,
            (numCells + 1) *
            sizeof(int)
        );

        computeForcesGPU(
            sortedR,
            cellStart,
            sortedIndex,
            F,
            r_min,
            numCells,
            d_sortedR,
            d_cellStart,
            d_F_sorted,
            FC
        );

        for (int i = 0; i < N; i++) {
            a_new[i] =
                F[i] / MASS;
        }

        for (int i = 0; i < N; i++) {
            v[i] +=
                0.5f *
                (a[i] + a_new[i]) *
                DT;
        }

        a.swap(a_new);

        if (step % CHECKPOINT == 0) {

            float KE =
                computeKE(v);

            float PE =
                computePE(
                    r,
                    sortedIndex,
                    cellStart,
                    r_min,
                    numCells,
                    FC
                );

            float E =
                KE + PE;

            float deltaE =
                E - E0;

            float minGap =
                minimumGap(sortedR);

            std::cout
                << std::setw(10)
                << step
                << std::setw(18)
                << step * DT
                << std::setw(18)
                << minGap
                << std::setw(18)
                << KE
                << std::setw(18)
                << PE
                << std::setw(18)
                << E
                << std::setw(18)
                << deltaE
                << '\n';
        }
    }

    cudaFree(d_sortedR);
    cudaFree(d_cellStart);
    cudaFree(d_F_sorted);

    return 0;
}

In [2]:
!nvcc -O2 md_fp32.cu -o md_fp32
!./md_fp32

DIAG sum(a^2) at t=0 = 70474.2  (mean a^2 = 1.07535)
Precision = FLOAT (FP32)
N = 65536
dt = 9.999999747e-05
Total simulated time = 1.999999949e-04
Initial KE = 3.277615070e+00
Initial PE = -2.100540234e+04
Initial E  = -2.100212500e+04

      Step              Time            MinGap                KE                PE           Total E              E-E0
         0   0.000000000e+00   1.101562500e+00   3.277615070e+00  -2.100540234e+04  -2.100212500e+04   0.000000000e+00
DIAG step 1: unchanged positions = 65514 / 65536
DIAG r[N-1] = 9.830235938e+04, ULP there = 7.812500000e-03, requested move v*dt+0.5*a*dt^2 = -1.010622782e-06


The chain is about 98,000 long. A float near 98,000 can only change in steps of 2^-7 = 0.0078, and a particle moves about 1e-6 per step, which is far below half of that, so `r[i] += ...` rounds back to the old value. After step 1, 65,514 of 65,536 positions are unchanged. The 22 that moved are presumably the particles near the origin, where the spacing between floats is small enough; the program does not print which ones.

With the positions frozen, the forces stay constant while the velocities keep integrating, so the kinetic energy grows like step^2 and the integrator creates energy. The numbers fit: sum(a^2) = 70,474 gives 0.5 x 70,474 x (1e-4)^2 = 3.5e-4 per step^2, which matches the coefficient of E - E0 / step^2 measured in the earlier 3000-step run. The constant `MinGap` of 1.1015625 in that run is 141/128, a multiple of 2^-7: it is the resolution of the float positions, not a physical gap.

So the 15% error measures storing absolute coordinates in `float`, not the quality of single-precision force arithmetic. Those two need to be separated.

## 2. Mixed precision

Positions, velocities, accelerations, the cell list and the energy sums stay in `double`. In the force kernel the position difference is taken in double (so the large coordinates cancel exactly) and only then converted to `float`; the pair-force arithmetic and the per-particle sum are in `float`. The result is converted back to double on the host.

In [3]:
%%writefile md_mixed.cu
#include <iostream>
#include <vector>
#include <random>
#include <cmath>
#include <iomanip>
#include <algorithm>
#include <cuda_runtime.h>

#define TILE_SIZE 256

const double SIGMA = 1.0;
const double EPSILON = 1.0;
const double MASS = 1.0;
const double RC = 2.5;
const double LCELL = 2.5;

const double DT = 0.0001;
const int N = 65536;
const int STEPS = 30000;
const int CHECKPOINT = 1000;

const float SIGMA_F = 1.0f;
const float EPSILON_F = 1.0f;
const float RC_F = 2.5f;

double ljForceD(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;
    return 24.0 * EPSILON * invR * (2.0 * sr12 - sr6);
}

double ljPotentialD(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;
    return 4.0 * EPSILON * (sr12 - sr6);
}

const double FC_D = ljForceD(RC);
const double UC_D = ljPotentialD(RC);
const float FC_F = (float)FC_D;

__global__ void cellListForceKernel(
    const double *sortedR,
    const int *cellStart,
    float *F_sorted,
    int N,
    int numCells,
    double r_min,
    float FC
) {
    int k = blockIdx.x * blockDim.x + threadIdx.x;

    if (k >= N) {
        return;
    }

    double myR = sortedR[k];

    int myCell = (int)floor((myR - r_min) / LCELL);

    if (myCell < 0) {
        myCell = 0;
    }

    if (myCell >= numCells) {
        myCell = numCells - 1;
    }

    float Fi = 0.0f;

    for (int c = myCell - 1; c <= myCell + 1; c++) {

        if (c < 0 || c >= numCells) {
            continue;
        }

        for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {

            if (m == k) {
                continue;
            }

            float dx = (float)(myR - sortedR[m]);
            float dist = fabsf(dx);

            if (dist < RC_F) {
                float invR = 1.0f / dist;
                float sr = SIGMA_F * invR;
                float sr6 = sr * sr * sr * sr * sr * sr;
                float sr12 = sr6 * sr6;

                float force =
                    24.0f * EPSILON_F * invR *
                    (2.0f * sr12 - sr6) - FC;

                Fi += force * (dx * invR);
            }
        }
    }

    F_sorted[k] = Fi;
}

void buildCellList(
    const std::vector<double>& r,
    std::vector<double>& sortedR,
    std::vector<int>& sortedIndex,
    std::vector<int>& cellStart,
    double& r_min,
    double& r_max,
    int& numCells
) {
    r_min = r[0];
    r_max = r[0];

    for (int i = 1; i < N; i++) {
        r_min = std::min(r_min, r[i]);
        r_max = std::max(r_max, r[i]);
    }

    numCells = (int)ceil((r_max - r_min) / LCELL);

    if (numCells < 1) {
        numCells = 1;
    }

    std::vector<int> cellCount(numCells, 0);

    for (int i = 0; i < N; i++) {
        int cell = (int)floor((r[i] - r_min) / LCELL);

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        cellCount[cell]++;
    }

    cellStart.assign(numCells + 1, 0);

    for (int c = 0; c < numCells; c++) {
        cellStart[c + 1] = cellStart[c] + cellCount[c];
    }

    std::vector<int> cursor = cellStart;

    sortedR.resize(N);
    sortedIndex.resize(N);

    for (int i = 0; i < N; i++) {
        int cell = (int)floor((r[i] - r_min) / LCELL);

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        int slot = cursor[cell];

        sortedR[slot] = r[i];
        sortedIndex[slot] = i;

        cursor[cell]++;
    }
}

void computeForcesGPU(
    const std::vector<double>& sortedR,
    const std::vector<int>& cellStart,
    const std::vector<int>& sortedIndex,
    std::vector<double>& F,
    double r_min,
    int numCells,
    double* d_sortedR,
    int* d_cellStart,
    float* d_F_sorted
) {
    cudaMemcpy(
        d_sortedR,
        sortedR.data(),
        N * sizeof(double),
        cudaMemcpyHostToDevice
    );

    cudaMemcpy(
        d_cellStart,
        cellStart.data(),
        (numCells + 1) * sizeof(int),
        cudaMemcpyHostToDevice
    );

    int blocks = (N + TILE_SIZE - 1) / TILE_SIZE;

    cellListForceKernel<<<blocks, TILE_SIZE>>>(
        d_sortedR,
        d_cellStart,
        d_F_sorted,
        N,
        numCells,
        r_min,
        FC_F
    );

    cudaError_t error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr
            << "Kernel launch error: "
            << cudaGetErrorString(error)
            << '\n';

        exit(1);
    }

    std::vector<float> F_sorted(N);

    cudaMemcpy(
        F_sorted.data(),
        d_F_sorted,
        N * sizeof(float),
        cudaMemcpyDeviceToHost
    );

    for (int k = 0; k < N; k++) {
        F[sortedIndex[k]] = (double)F_sorted[k];
    }
}

double computeKE(const std::vector<double>& v) {
    double KE = 0.0;

    for (int i = 0; i < N; i++) {
        KE += 0.5 * MASS * v[i] * v[i];
    }

    return KE;
}

double computePE(
    const std::vector<double>& r,
    const std::vector<int>& sortedIndex,
    const std::vector<int>& cellStart,
    double r_min,
    int numCells
) {
    double PE = 0.0;

    for (int i = 0; i < N; i++) {

        int myCell = (int)floor((r[i] - r_min) / LCELL);

        if (myCell < 0) {
            myCell = 0;
        }

        if (myCell >= numCells) {
            myCell = numCells - 1;
        }

        for (int c = myCell - 1; c <= myCell + 1; c++) {

            if (c < 0 || c >= numCells) {
                continue;
            }

            for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {

                int j = sortedIndex[m];

                if (j <= i) {
                    continue;
                }

                double dist = fabs(r[i] - r[j]);

                if (dist < RC) {
                    PE += ljPotentialD(dist) - UC_D
                          + FC_D * (dist - RC);
                }
            }
        }
    }

    return PE;
}

double minimumGap(const std::vector<double>& sortedR) {
    double minDist = 1e30;

    for (int i = 1; i < N; i++) {
        double gap = sortedR[i] - sortedR[i - 1];

        if (gap < minDist) {
            minDist = gap;
        }
    }

    return minDist;
}

int main() {

    std::vector<double> r(N);
    std::vector<double> v(N);
    std::vector<double> a(N);
    std::vector<double> a_new(N);
    std::vector<double> F(N);

    double spacing = 1.5;
    double jitterAmplitude = 0.2;

    std::mt19937 gen(42);

    std::uniform_real_distribution<double> jitter(
        -jitterAmplitude,
        jitterAmplitude
    );

    for (int i = 0; i < N; i++) {
        r[i] = i * spacing + jitter(gen);
    }

    for (int i = 0; i < N; i++) {
        v[i] = (i % 2 == 0) ? 0.01 : -0.01;
    }

    std::vector<double> sortedR;
    std::vector<int> sortedIndex;
    std::vector<int> cellStart;

    double r_min;
    double r_max;
    int numCells;

    buildCellList(r, sortedR, sortedIndex, cellStart, r_min, r_max, numCells);

    double* d_sortedR;
    int* d_cellStart;
    float* d_F_sorted;

    cudaMalloc(&d_sortedR, N * sizeof(double));
    cudaMalloc(&d_cellStart, (numCells + 1) * sizeof(int));
    cudaMalloc(&d_F_sorted, N * sizeof(float));

    computeForcesGPU(
        sortedR, cellStart, sortedIndex, F,
        r_min, numCells,
        d_sortedR, d_cellStart, d_F_sorted
    );

    for (int i = 0; i < N; i++) {
        a[i] = F[i] / MASS;
    }

    double KE0 = computeKE(v);
    double PE0 = computePE(r, sortedIndex, cellStart, r_min, numCells);
    double E0 = KE0 + PE0;

    std::cout << std::scientific << std::setprecision(9);

    std::cout << "Precision = MIXED (state/diagnostics FP64, pair-force arithmetic FP32)\n";
    std::cout << "N = " << N << '\n';
    std::cout << "dt = " << DT << '\n';
    std::cout << "Total simulated time = " << DT * STEPS << '\n';
    std::cout << "Initial KE = " << KE0 << '\n';
    std::cout << "Initial PE = " << PE0 << '\n';
    std::cout << "Initial E  = " << E0 << '\n';
    std::cout << '\n';

    std::cout
        << std::setw(10) << "Step"
        << std::setw(18) << "Time"
        << std::setw(18) << "MinGap"
        << std::setw(18) << "KE"
        << std::setw(18) << "PE"
        << std::setw(18) << "Total E"
        << std::setw(18) << "E-E0"
        << std::setw(18) << "|E-E0|/|E0|"
        << '\n';

    std::cout
        << std::setw(10) << 0
        << std::setw(18) << 0.0
        << std::setw(18) << minimumGap(sortedR)
        << std::setw(18) << KE0
        << std::setw(18) << PE0
        << std::setw(18) << E0
        << std::setw(18) << 0.0
        << std::setw(18) << 0.0
        << '\n';

    for (int step = 1; step <= STEPS; step++) {

        std::vector<double> r_before;
        if (step == 1) {
            r_before = r;
        }

        for (int i = 0; i < N; i++) {
            r[i] += v[i] * DT + 0.5 * a[i] * DT * DT;
        }

        if (step == 1) {
            int unchanged = 0;
            for (int i = 0; i < N; i++) {
                if (r[i] == r_before[i]) {
                    unchanged++;
                }
            }
            std::cout << "DIAG step 1: unchanged positions = " << unchanged
                      << " / " << N << "  (FP32 run had 65514)\n";
        }

        buildCellList(r, sortedR, sortedIndex, cellStart, r_min, r_max, numCells);

        cudaFree(d_cellStart);
        cudaMalloc(&d_cellStart, (numCells + 1) * sizeof(int));

        computeForcesGPU(
            sortedR, cellStart, sortedIndex, F,
            r_min, numCells,
            d_sortedR, d_cellStart, d_F_sorted
        );

        for (int i = 0; i < N; i++) {
            a_new[i] = F[i] / MASS;
        }

        for (int i = 0; i < N; i++) {
            v[i] += 0.5 * (a[i] + a_new[i]) * DT;
        }

        a.swap(a_new);

        if (step % CHECKPOINT == 0) {

            double KE = computeKE(v);
            double PE = computePE(r, sortedIndex, cellStart, r_min, numCells);
            double E = KE + PE;
            double deltaE = E - E0;
            double minGap = minimumGap(sortedR);

            std::cout
                << std::setw(10) << step
                << std::setw(18) << step * DT
                << std::setw(18) << minGap
                << std::setw(18) << KE
                << std::setw(18) << PE
                << std::setw(18) << E
                << std::setw(18) << deltaE
                << std::setw(18) << std::fabs(deltaE) / std::fabs(E0)
                << '\n';
        }
    }

    cudaFree(d_sortedR);
    cudaFree(d_cellStart);
    cudaFree(d_F_sorted);

    return 0;
}

In [4]:
!nvcc -O2 md_mixed.cu -o md_mixed
!./md_mixed

Precision = MIXED (state/diagnostics FP64, pair-force arithmetic FP32)
N = 65536
dt = 1.000000000e-04
Total simulated time = 3.000000000e+00
Initial KE = 3.276800000e+00
Initial PE = -2.097802761e+04
Initial E  = -2.097475081e+04

      Step              Time            MinGap                KE                PE           Total E              E-E0       |E-E0|/|E0|
         0   0.000000000e+00   1.100933961e+00   3.276800000e+00  -2.097802761e+04  -2.097475081e+04   0.000000000e+00   0.000000000e+00
DIAG step 1: unchanged positions = 0 / 65536  (FP32 run had 65514)
      1000   1.000000000e-01   1.116006447e+00   3.604009280e+02  -2.133515173e+04  -2.097475080e+04   4.522116797e-07   2.155981179e-11
      2000   2.000000000e-01   1.118257329e+00   1.453690065e+03  -2.242844090e+04  -2.097475083e+04  -2.646357461e-05   1.261687200e-09
      3000   3.000000000e-01   1.076835433e+00   2.956640306e+03  -2.393139127e+04  -2.097475097e+04  -1.623537646e-04   7.740438307e-09
      4000   4.00

No positions freeze, and the relative energy error is 3.7e-8 at t = 3, essentially the FP64 value. The energy oscillates in the same band as the FP64 run from part 1 (about -7.2e-4 to -8.2e-4 in absolute terms, against -6.2e-4 to -7.2e-4 for FP64).

## 3. Kernel benchmark

The FP64 and mixed force kernels on identical input, with force accuracy measured against the FP64 result.

In [5]:
%%writefile md_kernel_bench.cu
#include <iostream>
#include <vector>
#include <random>
#include <cmath>
#include <iomanip>
#include <algorithm>
#include <cuda_runtime.h>

#define TILE_SIZE 256

const double LCELL = 2.5;
const double RC = 2.5;
const int N = 65536;
const int WARMUP = 1;
const int RUNS = 20;

double ljForceD(double r) {
    double invR = 1.0 / r;
    double sr6 = invR * invR * invR * invR * invR * invR;
    double sr12 = sr6 * sr6;
    return 24.0 * invR * (2.0 * sr12 - sr6);
}

__global__ void kernelFP64(
    const double *sortedR,
    const int *cellStart,
    double *F_sorted,
    int N,
    int numCells,
    double r_min,
    double FC
) {
    int k = blockIdx.x * blockDim.x + threadIdx.x;
    if (k >= N) return;

    double myR = sortedR[k];
    int myCell = (int)floor((myR - r_min) / LCELL);
    if (myCell < 0) myCell = 0;
    if (myCell >= numCells) myCell = numCells - 1;

    double Fi = 0.0;

    for (int c = myCell - 1; c <= myCell + 1; c++) {
        if (c < 0 || c >= numCells) continue;

        for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {
            if (m == k) continue;

            double dx = myR - sortedR[m];
            double dist = fabs(dx);

            if (dist < RC) {
                double invR = 1.0 / dist;
                double sr = invR;
                double sr6 = sr * sr * sr * sr * sr * sr;
                double sr12 = sr6 * sr6;
                double force = 24.0 * invR * (2.0 * sr12 - sr6) - FC;
                Fi += force * (dx * invR);
            }
        }
    }

    F_sorted[k] = Fi;
}

__global__ void kernelMixed(
    const double *sortedR,
    const int *cellStart,
    float *F_sorted,
    int N,
    int numCells,
    double r_min,
    float FC
) {
    int k = blockIdx.x * blockDim.x + threadIdx.x;
    if (k >= N) return;

    double myR = sortedR[k];
    int myCell = (int)floor((myR - r_min) / LCELL);
    if (myCell < 0) myCell = 0;
    if (myCell >= numCells) myCell = numCells - 1;

    float Fi = 0.0f;

    for (int c = myCell - 1; c <= myCell + 1; c++) {
        if (c < 0 || c >= numCells) continue;

        for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {
            if (m == k) continue;

            float dx = (float)(myR - sortedR[m]);
            float dist = fabsf(dx);

            if (dist < 2.5f) {
                float invR = 1.0f / dist;
                float sr = invR;
                float sr6 = sr * sr * sr * sr * sr * sr;
                float sr12 = sr6 * sr6;
                float force = 24.0f * invR * (2.0f * sr12 - sr6) - FC;
                Fi += force * (dx * invR);
            }
        }
    }

    F_sorted[k] = Fi;
}

void buildCellList(
    const std::vector<double>& r,
    std::vector<double>& sortedR,
    std::vector<int>& cellStart,
    double& r_min,
    int& numCells
) {
    double r_max = r[0];
    r_min = r[0];
    for (int i = 1; i < N; i++) {
        r_min = std::min(r_min, r[i]);
        r_max = std::max(r_max, r[i]);
    }

    numCells = (int)ceil((r_max - r_min) / LCELL);
    if (numCells < 1) numCells = 1;

    auto cellOf = [&](double x) {
        int c = (int)floor((x - r_min) / LCELL);
        if (c < 0) c = 0;
        if (c >= numCells) c = numCells - 1;
        return c;
    };

    std::vector<int> cellCount(numCells, 0);
    for (int i = 0; i < N; i++) cellCount[cellOf(r[i])]++;

    cellStart.assign(numCells + 1, 0);
    for (int c = 0; c < numCells; c++)
        cellStart[c + 1] = cellStart[c] + cellCount[c];

    std::vector<int> cursor = cellStart;
    sortedR.resize(N);
    for (int i = 0; i < N; i++) sortedR[cursor[cellOf(r[i])]++] = r[i];
}

template <typename Launch>
void timeKernel(const char* name, Launch launch, double& minMs, double& steadyMs) {
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    for (int i = 0; i < WARMUP; i++) launch();
    cudaDeviceSynchronize();

    std::vector<double> t(RUNS);
    for (int i = 0; i < RUNS; i++) {
        cudaEventRecord(start);
        launch();
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);
        float ms;
        cudaEventElapsedTime(&ms, start, stop);
        t[i] = ms;
    }

    minMs = *std::min_element(t.begin(), t.end());
    double sum = 0;
    for (int i = RUNS / 2; i < RUNS; i++) sum += t[i];
    steadyMs = sum / (RUNS - RUNS / 2);

    std::cout << std::left << std::setw(10) << name << std::right
              << "  min = " << std::fixed << std::setprecision(6) << minMs
              << " ms   steady mean (last " << RUNS - RUNS / 2 << ") = "
              << steadyMs << " ms\n";

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
}

int main() {
    std::vector<double> r(N);
    std::mt19937 gen(42);
    std::uniform_real_distribution<double> jitter(-0.2, 0.2);
    for (int i = 0; i < N; i++) r[i] = i * 1.5 + jitter(gen);

    std::vector<double> sortedR;
    std::vector<int> cellStart;
    double r_min;
    int numCells;
    buildCellList(r, sortedR, cellStart, r_min, numCells);

    const double FC_D = ljForceD(RC);
    const float FC_F = (float)FC_D;

    double *d_R;
    int *d_cs;
    double *d_FA;
    float *d_FB;
    cudaMalloc(&d_R, N * sizeof(double));
    cudaMalloc(&d_cs, (numCells + 1) * sizeof(int));
    cudaMalloc(&d_FA, N * sizeof(double));
    cudaMalloc(&d_FB, N * sizeof(float));

    cudaMemcpy(d_R, sortedR.data(), N * sizeof(double), cudaMemcpyHostToDevice);
    cudaMemcpy(d_cs, cellStart.data(), (numCells + 1) * sizeof(int), cudaMemcpyHostToDevice);

    int blocks = (N + TILE_SIZE - 1) / TILE_SIZE;

    kernelFP64<<<blocks, TILE_SIZE>>>(d_R, d_cs, d_FA, N, numCells, r_min, FC_D);
    kernelMixed<<<blocks, TILE_SIZE>>>(d_R, d_cs, d_FB, N, numCells, r_min, FC_F);
    cudaError_t err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        std::cerr << "CUDA error: " << cudaGetErrorString(err) << '\n';
        return 1;
    }

    std::vector<double> FA(N);
    std::vector<float> FB(N);
    cudaMemcpy(FA.data(), d_FA, N * sizeof(double), cudaMemcpyDeviceToHost);
    cudaMemcpy(FB.data(), d_FB, N * sizeof(float), cudaMemcpyDeviceToHost);

    double maxAbs = 0, sumSq = 0, sumSqF = 0, maxF = 0;
    int worst = 0;
    for (int k = 0; k < N; k++) {
        double d = std::fabs(FA[k] - (double)FB[k]);
        if (d > maxAbs) { maxAbs = d; worst = k; }
        sumSq += d * d;
        sumSqF += FA[k] * FA[k];
        maxF = std::max(maxF, std::fabs(FA[k]));
    }

    std::cout << std::scientific << std::setprecision(6);
    std::cout << "N = " << N << ", numCells = " << numCells << "\n\n";
    std::cout << "FORCE ACCURACY (FP64 kernel = reference)\n";
    std::cout << "  rms |F| (FP64)        = " << std::sqrt(sumSqF / N) << '\n';
    std::cout << "  max |F| (FP64)        = " << maxF << '\n';
    std::cout << "  max |dF|              = " << maxAbs
              << "  (at sorted index " << worst << ", FP64 F = " << FA[worst] << ")\n";
    std::cout << "  rms |dF|              = " << std::sqrt(sumSq / N) << '\n';
    std::cout << "  max |dF| / rms |F|    = " << maxAbs / std::sqrt(sumSqF / N) << "\n\n";

    std::cout << "KERNEL TIMING (force kernel only, no memcpy)\n";
    double minA, steadyA, minB, steadyB;

    timeKernel("FP64", [&]() {
        kernelFP64<<<blocks, TILE_SIZE>>>(d_R, d_cs, d_FA, N, numCells, r_min, FC_D);
    }, minA, steadyA);

    timeKernel("Mixed", [&]() {
        kernelMixed<<<blocks, TILE_SIZE>>>(d_R, d_cs, d_FB, N, numCells, r_min, FC_F);
    }, minB, steadyB);

    std::cout << std::fixed << std::setprecision(3);
    std::cout << "\n  speed ratio FP64/Mixed (min)    = " << minA / minB << "x\n";
    std::cout << "  speed ratio FP64/Mixed (steady) = " << steadyA / steadyB << "x\n";

    cudaFree(d_R);
    cudaFree(d_cs);
    cudaFree(d_FA);
    cudaFree(d_FB);
    return 0;
}

In [6]:
!nvcc -O2 md_kernel_bench.cu -o md_kernel_bench
!./md_kernel_bench

N = 65536, numCells = 39321

FORCE ACCURACY (FP64 kernel = reference)
  rms |F| (FP64)        = 1.035518e+00
  max |F| (FP64)        = 2.445831e+00
  max |dF|              = 7.059334e-06  (at sorted index 59449, FP64 F = 2.130174e+00)
  rms |dF|              = 4.553590e-07
  max |dF| / rms |F|    = 6.817203e-06

KERNEL TIMING (force kernel only, no memcpy)
FP64        min = 0.202752 ms   steady mean (last 10) = 0.205530 ms
Mixed       min = 0.044704 ms   steady mean (last 10) = 0.045210 ms

  speed ratio FP64/Mixed (min)    = 4.535x
  speed ratio FP64/Mixed (steady) = 4.546x


The mixed kernel is 4.5x faster. Both kernels read the same `double` positions and differ only in the arithmetic, so a 4.5x change from precision alone means the kernel is limited by FP64 arithmetic and not by launch overhead. That applies to the cell-list kernel of part 1. The speedup is well short of 32x because the mixed kernel still loads doubles, subtracts positions in double, and computes the cell index in double.

Force error against the FP64 kernel: rms 4.6e-7 (relative to an rms force of 1.04), maximum 7.1e-6.

## 4. Where the time goes

A 4.5x faster kernel only matters if the kernel dominates a time step. Here is the per-phase timing of the full mixed-precision loop, with the cell list built on the host as in part 1.

In [7]:
%%writefile md_mixed_timed.cu
#include <iostream>
#include <vector>
#include <random>
#include <cmath>
#include <iomanip>
#include <algorithm>
#include <chrono>
#include <cuda_runtime.h>

#define TILE_SIZE 256

const double SIGMA = 1.0;
const double EPSILON = 1.0;
const double MASS = 1.0;
const double RC = 2.5;
const double LCELL = 2.5;

const double DT = 0.0001;
const int N = 65536;
const int STEPS = 1000;
const int CHECKPOINT = 500;
const int WARMUP_STEPS = 10;

const float SIGMA_F = 1.0f;
const float EPSILON_F = 1.0f;
const float RC_F = 2.5f;

double ljForceD(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;
    return 24.0 * EPSILON * invR * (2.0 * sr12 - sr6);
}

double ljPotentialD(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;
    return 4.0 * EPSILON * (sr12 - sr6);
}

const double FC_D = ljForceD(RC);
const double UC_D = ljPotentialD(RC);
const float FC_F = (float)FC_D;

__global__ void cellListForceKernel(
    const double *sortedR,
    const int *cellStart,
    float *F_sorted,
    int N,
    int numCells,
    double r_min,
    float FC
) {
    int k = blockIdx.x * blockDim.x + threadIdx.x;

    if (k >= N) {
        return;
    }

    double myR = sortedR[k];

    int myCell = (int)floor((myR - r_min) / LCELL);

    if (myCell < 0) {
        myCell = 0;
    }

    if (myCell >= numCells) {
        myCell = numCells - 1;
    }

    float Fi = 0.0f;

    for (int c = myCell - 1; c <= myCell + 1; c++) {

        if (c < 0 || c >= numCells) {
            continue;
        }

        for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {

            if (m == k) {
                continue;
            }

            float dx = (float)(myR - sortedR[m]);
            float dist = fabsf(dx);

            if (dist < RC_F) {
                float invR = 1.0f / dist;
                float sr = SIGMA_F * invR;
                float sr6 = sr * sr * sr * sr * sr * sr;
                float sr12 = sr6 * sr6;

                float force =
                    24.0f * EPSILON_F * invR *
                    (2.0f * sr12 - sr6) - FC;

                Fi += force * (dx * invR);
            }
        }
    }

    F_sorted[k] = Fi;
}

void buildCellList(
    const std::vector<double>& r,
    std::vector<double>& sortedR,
    std::vector<int>& sortedIndex,
    std::vector<int>& cellStart,
    double& r_min,
    double& r_max,
    int& numCells
) {
    r_min = r[0];
    r_max = r[0];

    for (int i = 1; i < N; i++) {
        r_min = std::min(r_min, r[i]);
        r_max = std::max(r_max, r[i]);
    }

    numCells = (int)ceil((r_max - r_min) / LCELL);

    if (numCells < 1) {
        numCells = 1;
    }

    std::vector<int> cellCount(numCells, 0);

    for (int i = 0; i < N; i++) {
        int cell = (int)floor((r[i] - r_min) / LCELL);

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        cellCount[cell]++;
    }

    cellStart.assign(numCells + 1, 0);

    for (int c = 0; c < numCells; c++) {
        cellStart[c + 1] = cellStart[c] + cellCount[c];
    }

    std::vector<int> cursor = cellStart;

    sortedR.resize(N);
    sortedIndex.resize(N);

    for (int i = 0; i < N; i++) {
        int cell = (int)floor((r[i] - r_min) / LCELL);

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        int slot = cursor[cell];

        sortedR[slot] = r[i];
        sortedIndex[slot] = i;

        cursor[cell]++;
    }
}

void computeForcesGPU(
    const std::vector<double>& sortedR,
    const std::vector<int>& cellStart,
    const std::vector<int>& sortedIndex,
    std::vector<double>& F,
    double r_min,
    int numCells,
    double* d_sortedR,
    int* d_cellStart,
    float* d_F_sorted
) {
    cudaMemcpy(d_sortedR, sortedR.data(), N * sizeof(double), cudaMemcpyHostToDevice);
    cudaMemcpy(d_cellStart, cellStart.data(), (numCells + 1) * sizeof(int), cudaMemcpyHostToDevice);

    int blocks = (N + TILE_SIZE - 1) / TILE_SIZE;

    cellListForceKernel<<<blocks, TILE_SIZE>>>(
        d_sortedR, d_cellStart, d_F_sorted, N, numCells, r_min, FC_F
    );

    cudaError_t error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr << "Kernel launch error: " << cudaGetErrorString(error) << '\n';
        exit(1);
    }

    std::vector<float> F_sorted(N);

    cudaMemcpy(F_sorted.data(), d_F_sorted, N * sizeof(float), cudaMemcpyDeviceToHost);

    for (int k = 0; k < N; k++) {
        F[sortedIndex[k]] = (double)F_sorted[k];
    }
}

double computeKE(const std::vector<double>& v) {
    double KE = 0.0;

    for (int i = 0; i < N; i++) {
        KE += 0.5 * MASS * v[i] * v[i];
    }

    return KE;
}

double computePE(
    const std::vector<double>& r,
    const std::vector<int>& sortedIndex,
    const std::vector<int>& cellStart,
    double r_min,
    int numCells
) {
    double PE = 0.0;

    for (int i = 0; i < N; i++) {

        int myCell = (int)floor((r[i] - r_min) / LCELL);

        if (myCell < 0) {
            myCell = 0;
        }

        if (myCell >= numCells) {
            myCell = numCells - 1;
        }

        for (int c = myCell - 1; c <= myCell + 1; c++) {

            if (c < 0 || c >= numCells) {
                continue;
            }

            for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {

                int j = sortedIndex[m];

                if (j <= i) {
                    continue;
                }

                double dist = fabs(r[i] - r[j]);

                if (dist < RC) {
                    PE += ljPotentialD(dist) - UC_D + FC_D * (dist - RC);
                }
            }
        }
    }

    return PE;
}

typedef std::chrono::high_resolution_clock Clock;

double msBetween(Clock::time_point a, Clock::time_point b) {
    return std::chrono::duration<double, std::milli>(b - a).count();
}

int main() {

    std::vector<double> r(N), v(N), a(N), a_new(N), F(N);

    std::mt19937 gen(42);
    std::uniform_real_distribution<double> jitter(-0.2, 0.2);

    for (int i = 0; i < N; i++) {
        r[i] = i * 1.5 + jitter(gen);
        v[i] = (i % 2 == 0) ? 0.01 : -0.01;
    }

    std::vector<double> sortedR;
    std::vector<int> sortedIndex;
    std::vector<int> cellStart;

    double r_min, r_max;
    int numCells;

    buildCellList(r, sortedR, sortedIndex, cellStart, r_min, r_max, numCells);

    double* d_sortedR;
    int* d_cellStart;
    float* d_F_sorted;

    cudaMalloc(&d_sortedR, N * sizeof(double));
    cudaMalloc(&d_cellStart, (numCells + 1) * sizeof(int));
    cudaMalloc(&d_F_sorted, N * sizeof(float));

    computeForcesGPU(sortedR, cellStart, sortedIndex, F, r_min, numCells,
                     d_sortedR, d_cellStart, d_F_sorted);

    for (int i = 0; i < N; i++) {
        a[i] = F[i] / MASS;
    }

    double E0 = computeKE(v) + computePE(r, sortedIndex, cellStart, r_min, numCells);

    std::cout << std::scientific << std::setprecision(6);
    std::cout << "Precision = MIXED, per-phase timing, N = " << N
              << ", steps = " << STEPS << " (first " << WARMUP_STEPS
              << " excluded)\n\n";

    double tPos = 0, tBuild = 0, tMalloc = 0, tH2D = 0, tKernel = 0,
           tD2H = 0, tUnsort = 0, tVel = 0, tStepTotal = 0;
    double tEnergy = 0;
    int timedSteps = 0;
    int energyEvals = 0;

    std::vector<float> F_sorted(N);
    int blocks = (N + TILE_SIZE - 1) / TILE_SIZE;

    auto wallStart = Clock::now();

    for (int step = 1; step <= STEPS; step++) {

        bool timed = step > WARMUP_STEPS;
        auto t0 = Clock::now();

        for (int i = 0; i < N; i++) {
            r[i] += v[i] * DT + 0.5 * a[i] * DT * DT;
        }
        auto t1 = Clock::now();

        buildCellList(r, sortedR, sortedIndex, cellStart, r_min, r_max, numCells);
        auto t2 = Clock::now();

        cudaFree(d_cellStart);
        cudaMalloc(&d_cellStart, (numCells + 1) * sizeof(int));
        auto t3 = Clock::now();

        cudaMemcpy(d_sortedR, sortedR.data(), N * sizeof(double), cudaMemcpyHostToDevice);
        cudaMemcpy(d_cellStart, cellStart.data(), (numCells + 1) * sizeof(int), cudaMemcpyHostToDevice);
        auto t4 = Clock::now();

        cellListForceKernel<<<blocks, TILE_SIZE>>>(
            d_sortedR, d_cellStart, d_F_sorted, N, numCells, r_min, FC_F
        );
        cudaError_t error = cudaGetLastError();
        if (error != cudaSuccess) {
            std::cerr << "Kernel launch error: " << cudaGetErrorString(error) << '\n';
            return 1;
        }
        cudaDeviceSynchronize();
        auto t5 = Clock::now();

        cudaMemcpy(F_sorted.data(), d_F_sorted, N * sizeof(float), cudaMemcpyDeviceToHost);
        auto t6 = Clock::now();

        for (int k = 0; k < N; k++) {
            F[sortedIndex[k]] = (double)F_sorted[k];
        }
        auto t7 = Clock::now();

        for (int i = 0; i < N; i++) {
            a_new[i] = F[i] / MASS;
        }
        for (int i = 0; i < N; i++) {
            v[i] += 0.5 * (a[i] + a_new[i]) * DT;
        }
        a.swap(a_new);
        auto t8 = Clock::now();

        if (timed) {
            tPos += msBetween(t0, t1);
            tBuild += msBetween(t1, t2);
            tMalloc += msBetween(t2, t3);
            tH2D += msBetween(t3, t4);
            tKernel += msBetween(t4, t5);
            tD2H += msBetween(t5, t6);
            tUnsort += msBetween(t6, t7);
            tVel += msBetween(t7, t8);
            tStepTotal += msBetween(t0, t8);
            timedSteps++;
        }

        if (step % CHECKPOINT == 0) {
            auto e0 = Clock::now();
            double KE = computeKE(v);
            double PE = computePE(r, sortedIndex, cellStart, r_min, numCells);
            auto e1 = Clock::now();
            tEnergy += msBetween(e0, e1);
            energyEvals++;

            double E = KE + PE;
            std::cout << "step " << std::setw(6) << step
                      << "   E-E0 = " << E - E0
                      << "   |E-E0|/|E0| = " << std::fabs(E - E0) / std::fabs(E0)
                      << '\n';
        }
    }

    auto wallEnd = Clock::now();

    std::cout << "\nPER-STEP PHASE TIMES (mean over " << timedSteps << " steps)\n";
    std::cout << std::fixed << std::setprecision(4);

    auto row = [&](const char* name, double total) {
        double perStep = total / timedSteps;
        std::cout << "  " << std::left << std::setw(26) << name << std::right
                  << std::setw(10) << perStep << " ms   "
                  << std::setw(6) << std::setprecision(2)
                  << 100.0 * total / tStepTotal << " %\n"
                  << std::setprecision(4);
    };

    row("position update (host)", tPos);
    row("buildCellList (host)", tBuild);
    row("cudaFree + cudaMalloc", tMalloc);
    row("H2D memcpy", tH2D);
    row("FORCE KERNEL (+sync)", tKernel);
    row("D2H memcpy", tD2H);
    row("un-sort (host)", tUnsort);
    row("accel + velocity (host)", tVel);

    std::cout << "  " << std::left << std::setw(26) << "TOTAL per step" << std::right
              << std::setw(10) << tStepTotal / timedSteps << " ms\n";

    std::cout << "\n  Energy diagnostic (KE+PE), per evaluation: "
              << tEnergy / std::max(1, energyEvals) << " ms  ("
              << energyEvals << " evals, not included in per-step total)\n";
    std::cout << "  Total wall time, all " << STEPS << " steps: "
              << msBetween(wallStart, wallEnd) / 1000.0 << " s\n";

    cudaFree(d_sortedR);
    cudaFree(d_cellStart);
    cudaFree(d_F_sorted);

    return 0;
}

In [8]:
!nvcc -O2 md_mixed_timed.cu -o md_mixed_timed
!./md_mixed_timed

Precision = MIXED, per-phase timing, N = 65536, steps = 1000 (first 10 excluded)

step    500   E-E0 = 4.096219e-07   |E-E0|/|E0| = 1.952928e-11
step   1000   E-E0 = 4.522117e-07   |E-E0|/|E0| = 2.155981e-11

PER-STEP PHASE TIMES (mean over 990 steps)
  position update (host)        0.1041 ms     6.02 %
  buildCellList (host)          1.0978 ms    63.49 %
  cudaFree + cudaMalloc         0.0155 ms     0.90 %
  H2D memcpy                    0.1969 ms    11.39 %
  FORCE KERNEL (+sync)          0.0624 ms     3.61 %
  D2H memcpy                    0.0592 ms     3.42 %
  un-sort (host)                0.0596 ms     3.45 %
  accel + velocity (host)       0.1335 ms     7.72 %
  TOTAL per step                1.7290 ms

  Energy diagnostic (KE+PE), per evaluation: 1.9419 ms  (2 evals, not included in per-step total)
  Total wall time, all 1000 steps: 1.7337 s


The force kernel is 3.6% of a step. Building the cell list on the host takes 63.5% and the data copies another 15%. Even an infinitely fast kernel would speed this loop up by only 1.04x.

## 5. Same source, FP64 vs mixed, three seeds

One source file builds either the pure FP64 kernel or the mixed kernel, so everything else is identical. Three initial seeds, 30,000 steps each.

In [9]:
%%writefile md_ab.cu
#include <iostream>
#include <vector>
#include <random>
#include <cmath>
#include <iomanip>
#include <algorithm>
#include <chrono>
#include <cuda_runtime.h>

#define TILE_SIZE 256

#ifndef SEED
#define SEED 42
#endif

#ifdef FP64_KERNEL
typedef double FT;
const char* MODE_NAME = "FP64 kernel (all double)";
#else
typedef float FT;
const char* MODE_NAME = "MIXED (state FP64, pair-force arithmetic FP32)";
#endif

const double SIGMA = 1.0;
const double EPSILON = 1.0;
const double MASS = 1.0;
const double RC = 2.5;
const double LCELL = 2.5;

const double DT = 0.0001;
const int N = 65536;
const int STEPS = 30000;
const int CHECKPOINT = 1000;
const int WARMUP_STEPS = 10;

double ljForceD(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;
    return 24.0 * EPSILON * invR * (2.0 * sr12 - sr6);
}

double ljPotentialD(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;
    return 4.0 * EPSILON * (sr12 - sr6);
}

const double FC_D = ljForceD(RC);
const double UC_D = ljPotentialD(RC);

template <typename T>
__global__ void cellListForceKernel(
    const double *sortedR,
    const int *cellStart,
    T *F_sorted,
    int N,
    int numCells,
    double r_min,
    T FC
) {
    int k = blockIdx.x * blockDim.x + threadIdx.x;

    if (k >= N) {
        return;
    }

    double myR = sortedR[k];

    int myCell = (int)floor((myR - r_min) / LCELL);

    if (myCell < 0) {
        myCell = 0;
    }

    if (myCell >= numCells) {
        myCell = numCells - 1;
    }

    T Fi = T(0);
    const T rc = T(2.5);

    for (int c = myCell - 1; c <= myCell + 1; c++) {

        if (c < 0 || c >= numCells) {
            continue;
        }

        for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {

            if (m == k) {
                continue;
            }

            T dx = (T)(myR - sortedR[m]);
            T dist = dx < T(0) ? -dx : dx;

            if (dist < rc) {
                T invR = T(1) / dist;
                T sr = invR;
                T sr6 = sr * sr * sr * sr * sr * sr;
                T sr12 = sr6 * sr6;

                T force =
                    T(24) * invR *
                    (T(2) * sr12 - sr6) - FC;

                Fi += force * (dx * invR);
            }
        }
    }

    F_sorted[k] = Fi;
}

void buildCellList(
    const std::vector<double>& r,
    std::vector<double>& sortedR,
    std::vector<int>& sortedIndex,
    std::vector<int>& cellStart,
    double& r_min,
    double& r_max,
    int& numCells
) {
    r_min = r[0];
    r_max = r[0];

    for (int i = 1; i < N; i++) {
        r_min = std::min(r_min, r[i]);
        r_max = std::max(r_max, r[i]);
    }

    numCells = (int)ceil((r_max - r_min) / LCELL);

    if (numCells < 1) {
        numCells = 1;
    }

    std::vector<int> cellCount(numCells, 0);

    for (int i = 0; i < N; i++) {
        int cell = (int)floor((r[i] - r_min) / LCELL);

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        cellCount[cell]++;
    }

    cellStart.assign(numCells + 1, 0);

    for (int c = 0; c < numCells; c++) {
        cellStart[c + 1] = cellStart[c] + cellCount[c];
    }

    std::vector<int> cursor = cellStart;

    sortedR.resize(N);
    sortedIndex.resize(N);

    for (int i = 0; i < N; i++) {
        int cell = (int)floor((r[i] - r_min) / LCELL);

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        int slot = cursor[cell];

        sortedR[slot] = r[i];
        sortedIndex[slot] = i;

        cursor[cell]++;
    }
}

double computeKE(const std::vector<double>& v) {
    double KE = 0.0;

    for (int i = 0; i < N; i++) {
        KE += 0.5 * MASS * v[i] * v[i];
    }

    return KE;
}

double computePE(
    const std::vector<double>& r,
    const std::vector<int>& sortedIndex,
    const std::vector<int>& cellStart,
    double r_min,
    int numCells
) {
    double PE = 0.0;

    for (int i = 0; i < N; i++) {

        int myCell = (int)floor((r[i] - r_min) / LCELL);

        if (myCell < 0) {
            myCell = 0;
        }

        if (myCell >= numCells) {
            myCell = numCells - 1;
        }

        for (int c = myCell - 1; c <= myCell + 1; c++) {

            if (c < 0 || c >= numCells) {
                continue;
            }

            for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {

                int j = sortedIndex[m];

                if (j <= i) {
                    continue;
                }

                double dist = fabs(r[i] - r[j]);

                if (dist < RC) {
                    PE += ljPotentialD(dist) - UC_D + FC_D * (dist - RC);
                }
            }
        }
    }

    return PE;
}

typedef std::chrono::high_resolution_clock Clock;

double msBetween(Clock::time_point a, Clock::time_point b) {
    return std::chrono::duration<double, std::milli>(b - a).count();
}

int main() {

    std::vector<double> r(N), v(N), a(N), a_new(N), F(N);

    std::mt19937 gen(SEED);
    std::uniform_real_distribution<double> jitter(-0.2, 0.2);

    for (int i = 0; i < N; i++) {
        r[i] = i * 1.5 + jitter(gen);
        v[i] = (i % 2 == 0) ? 0.01 : -0.01;
    }

    std::vector<double> sortedR;
    std::vector<int> sortedIndex;
    std::vector<int> cellStart;

    double r_min, r_max;
    int numCells;

    buildCellList(r, sortedR, sortedIndex, cellStart, r_min, r_max, numCells);

    double* d_sortedR;
    int* d_cellStart;
    FT* d_F_sorted;

    cudaMalloc(&d_sortedR, N * sizeof(double));
    cudaMalloc(&d_cellStart, (numCells + 1) * sizeof(int));
    cudaMalloc(&d_F_sorted, N * sizeof(FT));

    std::vector<FT> F_sorted(N);
    int blocks = (N + TILE_SIZE - 1) / TILE_SIZE;
    const FT FC = (FT)FC_D;

    cudaMemcpy(d_sortedR, sortedR.data(), N * sizeof(double), cudaMemcpyHostToDevice);
    cudaMemcpy(d_cellStart, cellStart.data(), (numCells + 1) * sizeof(int), cudaMemcpyHostToDevice);
    cellListForceKernel<FT><<<blocks, TILE_SIZE>>>(
        d_sortedR, d_cellStart, d_F_sorted, N, numCells, r_min, FC
    );
    cudaMemcpy(F_sorted.data(), d_F_sorted, N * sizeof(FT), cudaMemcpyDeviceToHost);
    for (int k = 0; k < N; k++) {
        F[sortedIndex[k]] = (double)F_sorted[k];
    }
    for (int i = 0; i < N; i++) {
        a[i] = F[i] / MASS;
    }

    double E0 = computeKE(v) + computePE(r, sortedIndex, cellStart, r_min, numCells);

    std::cout << std::scientific << std::setprecision(9);
    std::cout << "MODE = " << MODE_NAME << "\n";
    std::cout << "SEED = " << SEED << ", N = " << N << ", dt = " << DT
              << ", steps = " << STEPS << "\n";
    std::cout << "E0 = " << E0 << "\n\n";
    std::cout << std::setw(8) << "step"
              << std::setw(20) << "E-E0"
              << std::setw(20) << "|E-E0|/|E0|" << "\n";

    double tPos = 0, tBuild = 0, tMalloc = 0, tH2D = 0, tKernel = 0,
           tD2H = 0, tUnsort = 0, tVel = 0, tStepTotal = 0;
    int timedSteps = 0;
    double maxRel = 0.0;
    double lastRel = 0.0;

    for (int step = 1; step <= STEPS; step++) {

        bool timed = step > WARMUP_STEPS;
        auto t0 = Clock::now();

        for (int i = 0; i < N; i++) {
            r[i] += v[i] * DT + 0.5 * a[i] * DT * DT;
        }
        auto t1 = Clock::now();

        buildCellList(r, sortedR, sortedIndex, cellStart, r_min, r_max, numCells);
        auto t2 = Clock::now();

        cudaFree(d_cellStart);
        cudaMalloc(&d_cellStart, (numCells + 1) * sizeof(int));
        auto t3 = Clock::now();

        cudaMemcpy(d_sortedR, sortedR.data(), N * sizeof(double), cudaMemcpyHostToDevice);
        cudaMemcpy(d_cellStart, cellStart.data(), (numCells + 1) * sizeof(int), cudaMemcpyHostToDevice);
        auto t4 = Clock::now();

        cellListForceKernel<FT><<<blocks, TILE_SIZE>>>(
            d_sortedR, d_cellStart, d_F_sorted, N, numCells, r_min, FC
        );
        cudaError_t error = cudaGetLastError();
        if (error != cudaSuccess) {
            std::cerr << "Kernel launch error: " << cudaGetErrorString(error) << '\n';
            return 1;
        }
        cudaDeviceSynchronize();
        auto t5 = Clock::now();

        cudaMemcpy(F_sorted.data(), d_F_sorted, N * sizeof(FT), cudaMemcpyDeviceToHost);
        auto t6 = Clock::now();

        for (int k = 0; k < N; k++) {
            F[sortedIndex[k]] = (double)F_sorted[k];
        }
        auto t7 = Clock::now();

        for (int i = 0; i < N; i++) {
            a_new[i] = F[i] / MASS;
        }
        for (int i = 0; i < N; i++) {
            v[i] += 0.5 * (a[i] + a_new[i]) * DT;
        }
        a.swap(a_new);
        auto t8 = Clock::now();

        if (timed) {
            tPos += msBetween(t0, t1);
            tBuild += msBetween(t1, t2);
            tMalloc += msBetween(t2, t3);
            tH2D += msBetween(t3, t4);
            tKernel += msBetween(t4, t5);
            tD2H += msBetween(t5, t6);
            tUnsort += msBetween(t6, t7);
            tVel += msBetween(t7, t8);
            tStepTotal += msBetween(t0, t8);
            timedSteps++;
        }

        if (step % CHECKPOINT == 0) {
            double E = computeKE(v) + computePE(r, sortedIndex, cellStart, r_min, numCells);
            double rel = std::fabs(E - E0) / std::fabs(E0);
            maxRel = std::max(maxRel, rel);
            lastRel = rel;

            std::cout << std::setw(8) << step
                      << std::setw(20) << E - E0
                      << std::setw(20) << rel << "\n";
        }
    }

    std::cout << "\nSUMMARY  MODE = " << MODE_NAME << ", SEED = " << SEED << "\n";
    std::cout << "  max |E-E0|/|E0| over run = " << maxRel << "\n";
    std::cout << "  final |E-E0|/|E0|        = " << lastRel << "\n\n";

    std::cout << std::fixed << std::setprecision(4);
    std::cout << "PER-STEP PHASE TIMES (mean over " << timedSteps << " steps, ms)\n";
    std::cout << "  position update        " << tPos / timedSteps << "\n";
    std::cout << "  buildCellList          " << tBuild / timedSteps << "\n";
    std::cout << "  cudaFree+cudaMalloc    " << tMalloc / timedSteps << "\n";
    std::cout << "  H2D memcpy             " << tH2D / timedSteps << "\n";
    std::cout << "  FORCE KERNEL (+sync)   " << tKernel / timedSteps << "\n";
    std::cout << "  D2H memcpy             " << tD2H / timedSteps << "\n";
    std::cout << "  un-sort                " << tUnsort / timedSteps << "\n";
    std::cout << "  accel + velocity       " << tVel / timedSteps << "\n";
    std::cout << "  TOTAL per step         " << tStepTotal / timedSteps << "\n";

    cudaFree(d_sortedR);
    cudaFree(d_cellStart);
    cudaFree(d_F_sorted);

    return 0;
}

In [10]:
!nvcc -O2 md_ab.cu -o mixed42 && ./mixed42 > mixed_42.txt
!nvcc -O2 -DFP64_KERNEL md_ab.cu -o fp64_42 && ./fp64_42 > fp64_42.txt
!nvcc -O2 -DSEED=7 md_ab.cu -o mixed7 && ./mixed7 > mixed_7.txt
!nvcc -O2 -DSEED=7 -DFP64_KERNEL md_ab.cu -o fp64_7 && ./fp64_7 > fp64_7.txt
!nvcc -O2 -DSEED=123 md_ab.cu -o mixed123 && ./mixed123 > mixed_123.txt
!nvcc -O2 -DSEED=123 -DFP64_KERNEL md_ab.cu -o fp64_123 && ./fp64_123 > fp64_123.txt
!tail -n 14 mixed_42.txt fp64_42.txt mixed_7.txt fp64_7.txt mixed_123.txt fp64_123.txt

==> mixed_42.txt <==
SUMMARY  MODE = MIXED (state FP64, pair-force arithmetic FP32), SEED = 42
  max |E-E0|/|E0| over run = 3.924344266e-08
  final |E-E0|/|E0|        = 3.693918421e-08

PER-STEP PHASE TIMES (mean over 29990 steps, ms)
  position update        0.1022
  buildCellList          1.0695
  cudaFree+cudaMalloc    0.0146
  H2D memcpy             0.1960
  FORCE KERNEL (+sync)   0.0701
  D2H memcpy             0.0594
  un-sort                0.0602
  accel + velocity       0.1253
  TOTAL per step         1.6973

==> fp64_42.txt <==
SUMMARY  MODE = FP64 kernel (all double), SEED = 42
  max |E-E0|/|E0| over run = 3.409710138e-08
  final |E-E0|/|E0|        = 3.190215322e-08

PER-STEP PHASE TIMES (mean over 29990 steps, ms)
  position update        0.1020
  buildCellList          1.0731
  cudaFree+cudaMalloc    0.0154
  H2D memcpy             0.1945
  FORCE KERNEL (+sync)   0.2880
  D2H memcpy             0.1122
  un-sort                0.0812
  accel + velocity       0.1340
  TOTAL 

| Seed | Max relative energy error, mixed | FP64 | Total time per step, mixed | FP64 | Force kernel, mixed | FP64 |
|---|---|---|---|---|---|---|
| 42 | 3.92e-8 | 3.41e-8 | 1.70 ms | 2.00 ms | 0.070 ms | 0.288 ms |
| 7 | 3.90e-8 | 3.43e-8 | 1.68 ms | 2.00 ms | 0.069 ms | 0.287 ms |
| 123 | 3.79e-8 | 3.44e-8 | 1.75 ms | 2.08 ms | 0.070 ms | 0.291 ms |

Mixed precision conserves energy 10 to 15% worse than FP64 on every seed, always in the same direction, so the penalty is real but small. End to end it is only 1.18x faster, as the phase table predicted. FP64's error hardly changes with the seed (3.41e-8 to 3.44e-8), which suggests that most of the 3e-8 comes from the model and time step and not from precision.

## 6. Keeping the state on the GPU

To expose the kernel speed-up the host work has to go. In this version positions, velocities and accelerations stay on the device, and the position update, cell index, cell offsets and velocity update are all kernels. Only the first and last positions go back to the host each step, to size the cell grid.

The cell list here has no sort. In a 1D chain the Lennard-Jones core stops particles from overtaking each other, so index order is already spatial order and the cell offsets can be found directly. A device flag records any step where that assumption breaks. This shortcut does not carry over to 3D.

In [11]:
%%writefile md_gpu.cu
#include <iostream>
#include <vector>
#include <numeric>
#include <random>
#include <cmath>
#include <iomanip>
#include <algorithm>
#include <cuda_runtime.h>

#define TILE_SIZE 256

#ifndef SEED
#define SEED 42
#endif

#ifdef FP64_KERNEL
typedef double FT;
const char* MODE_NAME = "GPU-RESIDENT, FP64 kernel (all double)";
#else
typedef float FT;
const char* MODE_NAME = "GPU-RESIDENT, MIXED (state FP64, pair-force arithmetic FP32)";
#endif

const double SIGMA = 1.0;
const double EPSILON = 1.0;
const double MASS = 1.0;
const double RC = 2.5;
const double LCELL = 2.5;

const double DT = 0.0001;
const int N = 65536;
const int STEPS = 30000;
const int CHECKPOINT = 1000;
const int WARMUP_STEPS = 10;

double ljForceD(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;
    return 24.0 * EPSILON * invR * (2.0 * sr12 - sr6);
}

double ljPotentialD(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;
    return 4.0 * EPSILON * (sr12 - sr6);
}

const double FC_D = ljForceD(RC);
const double UC_D = ljPotentialD(RC);

__global__ void positionKernel(
    double *r, const double *v, const double *a, int N, double dt
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i >= N) {
        return;
    }

    r[i] += v[i] * dt + 0.5 * a[i] * dt * dt;
}

__global__ void cellIndexKernel(
    const double *r, int *cell, int *flag, int N, int numCells, double r0
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i >= N) {
        return;
    }

    int c = (int)floor((r[i] - r0) / LCELL);

    if (c < 0) {
        c = 0;
    }

    if (c >= numCells) {
        c = numCells - 1;
    }

    cell[i] = c;

    if (i > 0 && r[i] < r[i - 1]) {
        atomicOr(flag, 1);
    }
}

__global__ void cellStartKernel(
    const int *cell, int *cellStart, int N, int numCells
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i >= N) {
        return;
    }

    int prev = (i == 0) ? -1 : cell[i - 1];
    int cur = cell[i];

    for (int c = prev + 1; c <= cur; c++) {
        cellStart[c] = i;
    }

    if (i == N - 1) {
        cellStart[numCells] = N;
    }
}

template <typename T>
__global__ void cellListForceKernel(
    const double *sortedR,
    const int *cellStart,
    T *F_sorted,
    int N,
    int numCells,
    double r_min,
    T FC
) {
    int k = blockIdx.x * blockDim.x + threadIdx.x;

    if (k >= N) {
        return;
    }

    double myR = sortedR[k];

    int myCell = (int)floor((myR - r_min) / LCELL);

    if (myCell < 0) {
        myCell = 0;
    }

    if (myCell >= numCells) {
        myCell = numCells - 1;
    }

    T Fi = T(0);
    const T rc = T(2.5);

    for (int c = myCell - 1; c <= myCell + 1; c++) {

        if (c < 0 || c >= numCells) {
            continue;
        }

        for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {

            if (m == k) {
                continue;
            }

            T dx = (T)(myR - sortedR[m]);
            T dist = dx < T(0) ? -dx : dx;

            if (dist < rc) {
                T invR = T(1) / dist;
                T sr = invR;
                T sr6 = sr * sr * sr * sr * sr * sr;
                T sr12 = sr6 * sr6;

                T force =
                    T(24) * invR *
                    (T(2) * sr12 - sr6) - FC;

                Fi += force * (dx * invR);
            }
        }
    }

    F_sorted[k] = Fi;
}

template <typename T>
__global__ void initAccelKernel(double *a, const T *F, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i >= N) {
        return;
    }

    a[i] = (double)F[i] / MASS;
}

template <typename T>
__global__ void velocityKernel(
    double *v, double *a, const T *F, int N, double dt
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i >= N) {
        return;
    }

    double a_new = (double)F[i] / MASS;

    v[i] += 0.5 * (a[i] + a_new) * dt;
    a[i] = a_new;
}

double computeKE(const std::vector<double>& v) {
    double KE = 0.0;

    for (int i = 0; i < N; i++) {
        KE += 0.5 * MASS * v[i] * v[i];
    }

    return KE;
}

double computePE(
    const std::vector<double>& r,
    const std::vector<int>& sortedIndex,
    const std::vector<int>& cellStart,
    double r_min,
    int numCells
) {
    double PE = 0.0;

    for (int i = 0; i < N; i++) {

        int myCell = (int)floor((r[i] - r_min) / LCELL);

        if (myCell < 0) {
            myCell = 0;
        }

        if (myCell >= numCells) {
            myCell = numCells - 1;
        }

        for (int c = myCell - 1; c <= myCell + 1; c++) {

            if (c < 0 || c >= numCells) {
                continue;
            }

            for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {

                int j = sortedIndex[m];

                if (j <= i) {
                    continue;
                }

                double dist = fabs(r[i] - r[j]);

                if (dist < RC) {
                    PE += ljPotentialD(dist) - UC_D + FC_D * (dist - RC);
                }
            }
        }
    }

    return PE;
}

int main() {

    std::vector<double> h_r(N), h_v(N);

    std::mt19937 gen(SEED);
    std::uniform_real_distribution<double> jitter(-0.2, 0.2);

    for (int i = 0; i < N; i++) {
        h_r[i] = i * 1.5 + jitter(gen);
        h_v[i] = (i % 2 == 0) ? 0.01 : -0.01;
    }

    double *d_r, *d_v, *d_a;
    FT *d_F;
    int *d_cell, *d_cellStart, *d_flag;

    cudaMalloc(&d_r, N * sizeof(double));
    cudaMalloc(&d_v, N * sizeof(double));
    cudaMalloc(&d_a, N * sizeof(double));
    cudaMalloc(&d_F, N * sizeof(FT));
    cudaMalloc(&d_cell, N * sizeof(int));
    cudaMalloc(&d_cellStart, (N + 2) * sizeof(int));
    cudaMalloc(&d_flag, sizeof(int));

    cudaMemcpy(d_r, h_r.data(), N * sizeof(double), cudaMemcpyHostToDevice);
    cudaMemcpy(d_v, h_v.data(), N * sizeof(double), cudaMemcpyHostToDevice);
    cudaMemset(d_flag, 0, sizeof(int));

    const int blocks = (N + TILE_SIZE - 1) / TILE_SIZE;
    const FT FC = (FT)FC_D;

    double r0 = 0.0;
    int numCells = 1;

    cudaEvent_t ev[6];
    for (int i = 0; i < 6; i++) {
        cudaEventCreate(&ev[i]);
    }

    auto buildAndForce = [&](cudaEvent_t* e) {
        double edge[2];

        cudaMemcpy(&edge[0], d_r, sizeof(double), cudaMemcpyDeviceToHost);
        cudaMemcpy(&edge[1], d_r + (N - 1), sizeof(double), cudaMemcpyDeviceToHost);

        r0 = edge[0];
        numCells = (int)ceil((edge[1] - edge[0]) / LCELL);

        if (numCells < 1) {
            numCells = 1;
        }

        if (numCells > N + 1) {
            std::cerr << "numCells = " << numCells << " exceeds allocation\n";
            exit(1);
        }

        if (e) cudaEventRecord(e[2]);

        cellIndexKernel<<<blocks, TILE_SIZE>>>(
            d_r, d_cell, d_flag, N, numCells, r0
        );
        cellStartKernel<<<blocks, TILE_SIZE>>>(
            d_cell, d_cellStart, N, numCells
        );

        if (e) cudaEventRecord(e[3]);

        cellListForceKernel<FT><<<blocks, TILE_SIZE>>>(
            d_r, d_cellStart, d_F, N, numCells, r0, FC
        );

        if (e) cudaEventRecord(e[4]);
    };

    buildAndForce(nullptr);
    initAccelKernel<FT><<<blocks, TILE_SIZE>>>(d_a, d_F, N);

    cudaError_t err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        std::cerr << "CUDA error during init: " << cudaGetErrorString(err) << '\n';
        return 1;
    }

    std::vector<double> r(N), v(N);
    std::vector<int> cellStart, sortedIndex(N);
    std::iota(sortedIndex.begin(), sortedIndex.end(), 0);

    auto energy = [&]() -> double {
        cudaMemcpy(r.data(), d_r, N * sizeof(double), cudaMemcpyDeviceToHost);
        cudaMemcpy(v.data(), d_v, N * sizeof(double), cudaMemcpyDeviceToHost);
        cellStart.resize(numCells + 1);
        cudaMemcpy(cellStart.data(), d_cellStart, (numCells + 1) * sizeof(int),
                   cudaMemcpyDeviceToHost);
        return computeKE(v) + computePE(r, sortedIndex, cellStart, r0, numCells);
    };

    double E0 = energy();

    std::cout << std::scientific << std::setprecision(9);
    std::cout << "MODE = " << MODE_NAME << "\n";
    std::cout << "SEED = " << SEED << ", N = " << N << ", dt = " << DT
              << ", steps = " << STEPS << "\n";
    std::cout << "E0 = " << E0 << "\n\n";
    std::cout << std::setw(8) << "step"
              << std::setw(20) << "E-E0"
              << std::setw(20) << "|E-E0|/|E0|"
              << std::setw(10) << "orderOK" << "\n";

    double tPos = 0, tEdge = 0, tCell = 0, tForce = 0, tVel = 0, tTotal = 0;
    int timedSteps = 0;
    double maxRel = 0.0;
    double lastRel = 0.0;
    bool orderViolated = false;

    for (int step = 1; step <= STEPS; step++) {

        cudaEventRecord(ev[0]);

        positionKernel<<<blocks, TILE_SIZE>>>(d_r, d_v, d_a, N, DT);
        cudaEventRecord(ev[1]);

        buildAndForce(ev);

        velocityKernel<FT><<<blocks, TILE_SIZE>>>(d_v, d_a, d_F, N, DT);
        cudaEventRecord(ev[5]);
        cudaEventSynchronize(ev[5]);

        if (step > WARMUP_STEPS) {
            float ms;
            cudaEventElapsedTime(&ms, ev[0], ev[1]); tPos += ms;
            cudaEventElapsedTime(&ms, ev[1], ev[2]); tEdge += ms;
            cudaEventElapsedTime(&ms, ev[2], ev[3]); tCell += ms;
            cudaEventElapsedTime(&ms, ev[3], ev[4]); tForce += ms;
            cudaEventElapsedTime(&ms, ev[4], ev[5]); tVel += ms;
            cudaEventElapsedTime(&ms, ev[0], ev[5]); tTotal += ms;
            timedSteps++;
        }

        if (step % CHECKPOINT == 0) {
            err = cudaGetLastError();
            if (err != cudaSuccess) {
                std::cerr << "CUDA error: " << cudaGetErrorString(err) << '\n';
                return 1;
            }

            int flag = 0;
            cudaMemcpy(&flag, d_flag, sizeof(int), cudaMemcpyDeviceToHost);
            if (flag) {
                orderViolated = true;
            }

            double E = energy();
            double rel = std::fabs(E - E0) / std::fabs(E0);
            maxRel = std::max(maxRel, rel);
            lastRel = rel;

            std::cout << std::setw(8) << step
                      << std::setw(20) << E - E0
                      << std::setw(20) << rel
                      << std::setw(10) << (flag ? "VIOLATED" : "ok") << "\n";
        }
    }

    std::cout << "\nSUMMARY  " << MODE_NAME << ", SEED = " << SEED << "\n";
    std::cout << "  max |E-E0|/|E0| over run = " << maxRel << "\n";
    std::cout << "  final |E-E0|/|E0|        = " << lastRel << "\n";
    std::cout << "  particle order preserved  = "
              << (orderViolated ? "NO (results invalid)" : "yes") << "\n\n";

    std::cout << std::fixed << std::setprecision(4);
    std::cout << "PER-STEP PHASE TIMES (CUDA events, mean over " << timedSteps << " steps, ms)\n";
    std::cout << "  position kernel                 " << tPos / timedSteps << "\n";
    std::cout << "  D2H r[0],r[N-1] + host numCells " << tEdge / timedSteps << "\n";
    std::cout << "  cellIndex + cellStart kernels   " << tCell / timedSteps << "\n";
    std::cout << "  FORCE KERNEL                    " << tForce / timedSteps << "\n";
    std::cout << "  velocity kernel                 " << tVel / timedSteps << "\n";
    std::cout << "  TOTAL per step                  " << tTotal / timedSteps << "\n";

    cudaFree(d_r);
    cudaFree(d_v);
    cudaFree(d_a);
    cudaFree(d_F);
    cudaFree(d_cell);
    cudaFree(d_cellStart);
    cudaFree(d_flag);

    return 0;
}

In [12]:
!nvcc -O2 md_gpu.cu -o md_gpu_mixed && ./md_gpu_mixed > gpu_mixed.txt
!nvcc -O2 -DFP64_KERNEL md_gpu.cu -o md_gpu_fp64 && ./md_gpu_fp64 > gpu_fp64.txt
!tail -n 16 gpu_mixed.txt gpu_fp64.txt

==> gpu_mixed.txt <==
   28000    -7.962164782e-04     3.796071217e-08        ok
   29000    -7.909718624e-04     3.771066793e-08        ok
   30000    -7.747934069e-04     3.693933788e-08        ok

SUMMARY  GPU-RESIDENT, MIXED (state FP64, pair-force arithmetic FP32), SEED = 42
  max |E-E0|/|E0| over run = 3.924395849e-08
  final |E-E0|/|E0|        = 3.693933788e-08
  particle order preserved  = yes

PER-STEP PHASE TIMES (CUDA events, mean over 29990 steps, ms)
  position kernel                 0.0086
  D2H r[0],r[N-1] + host numCells 0.0219
  cellIndex + cellStart kernels   0.0162
  FORCE KERNEL                    0.0211
  velocity kernel                 0.0067
  TOTAL per step                  0.0745

==> gpu_fp64.txt <==
   28000    -6.904300426e-04     3.291719883e-08        ok
   29000    -6.835945969e-04     3.259130958e-08        ok
   30000    -6.691391791e-04     3.190212772e-08        ok

SUMMARY  GPU-RESIDENT, FP64 kernel (all double), SEED = 42
  max |E-E0|/|E0| over run 

| ms per step (seed 42) | Mixed | FP64 |
|---|---|---|
| Host-driven loop (section 5) | 1.697 | 2.000 |
| GPU-resident | 0.0745 | 0.1531 |
| Speedup | 22.8x | 13.1x |

Mixed is now 2.06x faster than FP64 end to end, up from 1.18x, and the force kernel is 28% of a step (it was 3.6%). Energy errors match the host-driven runs to five or six digits (3.924e-8 and 3.410e-8 maximum).

A second variant computes the grid size on the device so that no host round trip is needed.

In [13]:
%%writefile md_gpu2.cu
#include <iostream>
#include <vector>
#include <numeric>
#include <random>
#include <cmath>
#include <iomanip>
#include <algorithm>
#include <cuda_runtime.h>

#define TILE_SIZE 256

#ifndef SEED
#define SEED 42
#endif

#ifdef FP64_KERNEL
typedef double FT;
const char* MODE_NAME = "GPU-RESIDENT v2 (no host sync), FP64 kernel (all double)";
#else
typedef float FT;
const char* MODE_NAME = "GPU-RESIDENT v2 (no host sync), MIXED (state FP64, pair-force FP32)";
#endif

const double SIGMA = 1.0;
const double EPSILON = 1.0;
const double MASS = 1.0;
const double RC = 2.5;
const double LCELL = 2.5;

const double DT = 0.0001;
const int N = 65536;
const int STEPS = 30000;
const int CHECKPOINT = 1000;
const int WARMUP_STEPS = 10;

double ljForceD(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;
    return 24.0 * EPSILON * invR * (2.0 * sr12 - sr6);
}

double ljPotentialD(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;
    return 4.0 * EPSILON * (sr12 - sr6);
}

const double FC_D = ljForceD(RC);
const double UC_D = ljPotentialD(RC);

__device__ __forceinline__ int numCellsDev(const double *r, int N) {
    int nc = (int)ceil((r[N - 1] - r[0]) / LCELL);
    return nc < 1 ? 1 : nc;
}

__global__ void positionKernel(
    double *r, const double *v, const double *a, int N, double dt
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i >= N) {
        return;
    }

    r[i] += v[i] * dt + 0.5 * a[i] * dt * dt;
}

__global__ void cellIndexKernel(
    const double *r, int *cell, int *flag, int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i >= N) {
        return;
    }

    int numCells = numCellsDev(r, N);

    if (numCells > N + 1) {
        if (i == 0) {
            atomicOr(flag, 2);
        }
        return;
    }

    double r0 = r[0];

    int c = (int)floor((r[i] - r0) / LCELL);

    if (c < 0) {
        c = 0;
    }

    if (c >= numCells) {
        c = numCells - 1;
    }

    cell[i] = c;

    if (i > 0 && r[i] < r[i - 1]) {
        atomicOr(flag, 1);
    }
}

__global__ void cellStartKernel(
    const double *r, const int *cell, int *cellStart, int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i >= N) {
        return;
    }

    int numCells = numCellsDev(r, N);

    if (numCells > N + 1) {
        return;
    }

    int prev = (i == 0) ? -1 : cell[i - 1];
    int cur = cell[i];

    for (int c = prev + 1; c <= cur; c++) {
        cellStart[c] = i;
    }

    if (i == N - 1) {
        cellStart[numCells] = N;
    }
}

template <typename T>
__global__ void cellListForceKernel(
    const double *sortedR,
    const int *cellStart,
    T *F_sorted,
    int N,
    T FC
) {
    int k = blockIdx.x * blockDim.x + threadIdx.x;

    if (k >= N) {
        return;
    }

    int numCells = numCellsDev(sortedR, N);

    if (numCells > N + 1) {
        return;
    }

    double r_min = sortedR[0];
    double myR = sortedR[k];

    int myCell = (int)floor((myR - r_min) / LCELL);

    if (myCell < 0) {
        myCell = 0;
    }

    if (myCell >= numCells) {
        myCell = numCells - 1;
    }

    T Fi = T(0);
    const T rc = T(2.5);

    for (int c = myCell - 1; c <= myCell + 1; c++) {

        if (c < 0 || c >= numCells) {
            continue;
        }

        for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {

            if (m == k) {
                continue;
            }

            T dx = (T)(myR - sortedR[m]);
            T dist = dx < T(0) ? -dx : dx;

            if (dist < rc) {
                T invR = T(1) / dist;
                T sr = invR;
                T sr6 = sr * sr * sr * sr * sr * sr;
                T sr12 = sr6 * sr6;

                T force =
                    T(24) * invR *
                    (T(2) * sr12 - sr6) - FC;

                Fi += force * (dx * invR);
            }
        }
    }

    F_sorted[k] = Fi;
}

template <typename T>
__global__ void initAccelKernel(double *a, const T *F, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i >= N) {
        return;
    }

    a[i] = (double)F[i] / MASS;
}

template <typename T>
__global__ void velocityKernel(
    double *v, double *a, const T *F, int N, double dt
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i >= N) {
        return;
    }

    double a_new = (double)F[i] / MASS;

    v[i] += 0.5 * (a[i] + a_new) * dt;
    a[i] = a_new;
}

double computeKE(const std::vector<double>& v) {
    double KE = 0.0;

    for (int i = 0; i < N; i++) {
        KE += 0.5 * MASS * v[i] * v[i];
    }

    return KE;
}

double computePE(
    const std::vector<double>& r,
    const std::vector<int>& sortedIndex,
    const std::vector<int>& cellStart,
    double r_min,
    int numCells
) {
    double PE = 0.0;

    for (int i = 0; i < N; i++) {

        int myCell = (int)floor((r[i] - r_min) / LCELL);

        if (myCell < 0) {
            myCell = 0;
        }

        if (myCell >= numCells) {
            myCell = numCells - 1;
        }

        for (int c = myCell - 1; c <= myCell + 1; c++) {

            if (c < 0 || c >= numCells) {
                continue;
            }

            for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {

                int j = sortedIndex[m];

                if (j <= i) {
                    continue;
                }

                double dist = fabs(r[i] - r[j]);

                if (dist < RC) {
                    PE += ljPotentialD(dist) - UC_D + FC_D * (dist - RC);
                }
            }
        }
    }

    return PE;
}

int main() {

    std::vector<double> h_r(N), h_v(N);

    std::mt19937 gen(SEED);
    std::uniform_real_distribution<double> jitter(-0.2, 0.2);

    for (int i = 0; i < N; i++) {
        h_r[i] = i * 1.5 + jitter(gen);
        h_v[i] = (i % 2 == 0) ? 0.01 : -0.01;
    }

    double *d_r, *d_v, *d_a;
    FT *d_F;
    int *d_cell, *d_cellStart, *d_flag;

    cudaMalloc(&d_r, N * sizeof(double));
    cudaMalloc(&d_v, N * sizeof(double));
    cudaMalloc(&d_a, N * sizeof(double));
    cudaMalloc(&d_F, N * sizeof(FT));
    cudaMalloc(&d_cell, N * sizeof(int));
    cudaMalloc(&d_cellStart, (N + 2) * sizeof(int));
    cudaMalloc(&d_flag, sizeof(int));

    cudaMemcpy(d_r, h_r.data(), N * sizeof(double), cudaMemcpyHostToDevice);
    cudaMemcpy(d_v, h_v.data(), N * sizeof(double), cudaMemcpyHostToDevice);
    cudaMemset(d_flag, 0, sizeof(int));

    const int blocks = (N + TILE_SIZE - 1) / TILE_SIZE;
    const FT FC = (FT)FC_D;

    cudaEvent_t ev[5];
    for (int i = 0; i < 5; i++) {
        cudaEventCreate(&ev[i]);
    }

    auto buildAndForce = [&](cudaEvent_t* e_afterCell, cudaEvent_t* e_afterForce) {
        cellIndexKernel<<<blocks, TILE_SIZE>>>(d_r, d_cell, d_flag, N);
        cellStartKernel<<<blocks, TILE_SIZE>>>(d_r, d_cell, d_cellStart, N);

        if (e_afterCell) cudaEventRecord(*e_afterCell);

        cellListForceKernel<FT><<<blocks, TILE_SIZE>>>(
            d_r, d_cellStart, d_F, N, FC
        );

        if (e_afterForce) cudaEventRecord(*e_afterForce);
    };

    buildAndForce(nullptr, nullptr);
    initAccelKernel<FT><<<blocks, TILE_SIZE>>>(d_a, d_F, N);

    cudaError_t err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        std::cerr << "CUDA error during init: " << cudaGetErrorString(err) << '\n';
        return 1;
    }

    std::vector<double> r(N), v(N);
    std::vector<int> cellStart, sortedIndex(N);
    std::iota(sortedIndex.begin(), sortedIndex.end(), 0);

    auto energy = [&]() -> double {
        cudaMemcpy(r.data(), d_r, N * sizeof(double), cudaMemcpyDeviceToHost);
        cudaMemcpy(v.data(), d_v, N * sizeof(double), cudaMemcpyDeviceToHost);

        int numCells = (int)ceil((r[N - 1] - r[0]) / LCELL);
        if (numCells < 1) {
            numCells = 1;
        }

        cellStart.resize(numCells + 1);
        cudaMemcpy(cellStart.data(), d_cellStart, (numCells + 1) * sizeof(int),
                   cudaMemcpyDeviceToHost);

        return computeKE(v) + computePE(r, sortedIndex, cellStart, r[0], numCells);
    };

    double E0 = energy();

    std::cout << std::scientific << std::setprecision(9);
    std::cout << "MODE = " << MODE_NAME << "\n";
    std::cout << "SEED = " << SEED << ", N = " << N << ", dt = " << DT
              << ", steps = " << STEPS << "\n";
    std::cout << "E0 = " << E0 << "\n\n";
    std::cout << std::setw(8) << "step"
              << std::setw(20) << "E-E0"
              << std::setw(20) << "|E-E0|/|E0|"
              << std::setw(10) << "flags" << "\n";

    double tPos = 0, tCell = 0, tForce = 0, tVel = 0, tTotal = 0;
    int timedSteps = 0;
    double maxRel = 0.0;
    double lastRel = 0.0;
    int flagsSeen = 0;

    for (int step = 1; step <= STEPS; step++) {

        cudaEventRecord(ev[0]);

        positionKernel<<<blocks, TILE_SIZE>>>(d_r, d_v, d_a, N, DT);
        cudaEventRecord(ev[1]);

        buildAndForce(&ev[2], &ev[3]);

        velocityKernel<FT><<<blocks, TILE_SIZE>>>(d_v, d_a, d_F, N, DT);
        cudaEventRecord(ev[4]);
        cudaEventSynchronize(ev[4]);

        if (step > WARMUP_STEPS) {
            float ms;
            cudaEventElapsedTime(&ms, ev[0], ev[1]); tPos += ms;
            cudaEventElapsedTime(&ms, ev[1], ev[2]); tCell += ms;
            cudaEventElapsedTime(&ms, ev[2], ev[3]); tForce += ms;
            cudaEventElapsedTime(&ms, ev[3], ev[4]); tVel += ms;
            cudaEventElapsedTime(&ms, ev[0], ev[4]); tTotal += ms;
            timedSteps++;
        }

        if (step % CHECKPOINT == 0) {
            err = cudaGetLastError();
            if (err != cudaSuccess) {
                std::cerr << "CUDA error: " << cudaGetErrorString(err) << '\n';
                return 1;
            }

            int flag = 0;
            cudaMemcpy(&flag, d_flag, sizeof(int), cudaMemcpyDeviceToHost);
            flagsSeen |= flag;

            double E = energy();
            double rel = std::fabs(E - E0) / std::fabs(E0);
            maxRel = std::max(maxRel, rel);
            lastRel = rel;

            std::cout << std::setw(8) << step
                      << std::setw(20) << E - E0
                      << std::setw(20) << rel
                      << std::setw(10) << flag << "\n";
        }
    }

    std::cout << "\nSUMMARY  " << MODE_NAME << ", SEED = " << SEED << "\n";
    std::cout << "  max |E-E0|/|E0| over run = " << maxRel << "\n";
    std::cout << "  final |E-E0|/|E0|        = " << lastRel << "\n";
    std::cout << "  device flags (0 = all ok) = " << flagsSeen << "\n\n";

    std::cout << std::fixed << std::setprecision(4);
    std::cout << "PER-STEP PHASE TIMES (CUDA events, mean over " << timedSteps << " steps, ms)\n";
    std::cout << "  position kernel                 " << tPos / timedSteps << "\n";
    std::cout << "  cellIndex + cellStart kernels   " << tCell / timedSteps << "\n";
    std::cout << "  FORCE KERNEL                    " << tForce / timedSteps << "\n";
    std::cout << "  velocity kernel                 " << tVel / timedSteps << "\n";
    std::cout << "  TOTAL per step                  " << tTotal / timedSteps << "\n";

    cudaFree(d_r);
    cudaFree(d_v);
    cudaFree(d_a);
    cudaFree(d_F);
    cudaFree(d_cell);
    cudaFree(d_cellStart);
    cudaFree(d_flag);

    return 0;
}

In [14]:
!nvcc -O2 md_gpu2.cu -o md_gpu2_mixed && ./md_gpu2_mixed > gpu2_mixed.txt
!nvcc -O2 -DFP64_KERNEL md_gpu2.cu -o md_gpu2_fp64 && ./md_gpu2_fp64 > gpu2_fp64.txt
!tail -n 14 gpu2_mixed.txt gpu2_fp64.txt

==> gpu2_mixed.txt <==
   29000    -7.909718624e-04     3.771066793e-08         0
   30000    -7.747934069e-04     3.693933788e-08         0

SUMMARY  GPU-RESIDENT v2 (no host sync), MIXED (state FP64, pair-force FP32), SEED = 42
  max |E-E0|/|E0| over run = 3.924395849e-08
  final |E-E0|/|E0|        = 3.693933788e-08
  device flags (0 = all ok) = 0

PER-STEP PHASE TIMES (CUDA events, mean over 29990 steps, ms)
  position kernel                 0.0088
  cellIndex + cellStart kernels   0.0270
  FORCE KERNEL                    0.0279
  velocity kernel                 0.0068
  TOTAL per step                  0.0705

==> gpu2_fp64.txt <==
   29000    -6.835945969e-04     3.259130958e-08         0
   30000    -6.691391791e-04     3.190212772e-08         0

SUMMARY  GPU-RESIDENT v2 (no host sync), FP64 kernel (all double), SEED = 42
  max |E-E0|/|E0| over run = 3.409710572e-08
  final |E-E0|/|E0|        = 3.190212772e-08
  device flags (0 = all ok) = 0

PER-STEP PHASE TIMES (CUDA events, mea

Removing the host round trip (0.022 ms) saved only 0.004 ms per step for mixed (0.0745 to 0.0705 ms) and 0.002 ms for FP64. The cell-list and force kernels together got slower by about 18 microseconds, most of the 22 that were saved. Every thread now repeats a double-precision divide to compute the grid size, which is my explanation, but I have not tested it: computing the grid size once in a single thread would settle it. Energy results are unchanged to every printed digit.

## 7. Validation against OpenMM

Everything above checks the code against itself. As an external check, the same model (shifted-force Lennard-Jones, cutoff 2.5, velocity Verlet, dt = 1e-4) was run in OpenMM 8.6.1 on its double-precision Reference platform. The chain is run as a collinear 3D system, so the y and z forces are exactly zero. N is 4,096 here, so the OpenMM run is quick.

The next cell dumps the t = 0 forces, the energy trajectory and the final positions of both the FP64 and the mixed GPU runs. `validation/compare_to_oracle.py` then compares them with OpenMM; it runs outside Kaggle because it needs OpenMM installed (`pip install openmm`). The tolerances in that script were fixed before looking at any result.

In [15]:
%%writefile md_dump.cu
  #include <iostream>
  #include <vector>
  #include <numeric>
  #include <random>
  #include <cmath>
  #include <cstdio>
  #include <cstdlib>
  #include <algorithm>
  #include <cuda_runtime.h>

  #define TILE_SIZE 256

  #ifndef NPART
  #define NPART 4096
  #endif

  #ifndef SEED
  #define SEED 42
  #endif

  const double SIGMA = 1.0;
  const double EPSILON = 1.0;
  const double MASS = 1.0;
  const double RC = 2.5;
  const double LCELL = 2.5;
  const double DT = 0.0001;

  const int N = NPART;
  const int STEPS = 30000;
  const int CHECKPOINT = 1000;

  double ljForceD(double r) {
      double invR = 1.0 / r;
      double sr = SIGMA * invR;
      double sr6 = sr * sr * sr * sr * sr * sr;
      double sr12 = sr6 * sr6;
      return 24.0 * EPSILON * invR * (2.0 * sr12 - sr6);
  }

  double ljPotentialD(double r) {
      double invR = 1.0 / r;
      double sr = SIGMA * invR;
      double sr6 = sr * sr * sr * sr * sr * sr;
      double sr12 = sr6 * sr6;
      return 4.0 * EPSILON * (sr12 - sr6);
  }

  const double FC_D = ljForceD(RC);
  const double UC_D = ljPotentialD(RC);

  struct GridInfo {
      double r0;
      int numCells;
  };

  __global__ void gridKernel(
      const double *r, GridInfo *g, int *flag, int N
  ) {
      double r0 = r[0];
      int nc = (int)ceil((r[N - 1] - r0) / LCELL);

      if (nc < 1) {
          nc = 1;
      }

      if (nc > N + 1) {
          atomicOr(flag, 2);
      }

      g->r0 = r0;
      g->numCells = nc;
  }

  __global__ void positionKernel(
      double *r, const double *v, const double *a, int N, double dt
  ) {
      int i = blockIdx.x * blockDim.x + threadIdx.x;

      if (i >= N) {
          return;
      }

      r[i] += v[i] * dt + 0.5 * a[i] * dt * dt;
  }

  __global__ void cellIndexKernel(
      const double *r, int *cell, int *flag, const GridInfo *g, int N
  ) {
      int i = blockIdx.x * blockDim.x + threadIdx.x;

      if (i >= N) {
          return;
      }

      int numCells = g->numCells;

      if (numCells > N + 1) {
          return;
      }

      int c = (int)floor((r[i] - g->r0) / LCELL);

      if (c < 0) {
          c = 0;
      }

      if (c >= numCells) {
          c = numCells - 1;
      }

      cell[i] = c;

      if (i > 0 && r[i] < r[i - 1]) {
          atomicOr(flag, 1);
      }
  }

  __global__ void cellStartKernel(
      const int *cell, int *cellStart, const GridInfo *g, int N
  ) {
      int i = blockIdx.x * blockDim.x + threadIdx.x;

      if (i >= N) {
          return;
      }

      int numCells = g->numCells;

      if (numCells > N + 1) {
          return;
      }

      int prev = (i == 0) ? -1 : cell[i - 1];
      int cur = cell[i];

      for (int c = prev + 1; c <= cur; c++) {
          cellStart[c] = i;
      }

      if (i == N - 1) {
          cellStart[numCells] = N;
      }
  }

  template <typename T>
  __global__ void cellListForceKernel(
      const double *sortedR,
      const int *cellStart,
      T *F_sorted,
      const GridInfo *g,
      int N,
      T FC
  ) {
      int k = blockIdx.x * blockDim.x + threadIdx.x;

      if (k >= N) {
          return;
      }

      int numCells = g->numCells;

      if (numCells > N + 1) {
          return;
      }

      double r_min = g->r0;
      double myR = sortedR[k];

      int myCell = (int)floor((myR - r_min) / LCELL);

      if (myCell < 0) {
          myCell = 0;
      }

      if (myCell >= numCells) {
          myCell = numCells - 1;
      }

      T Fi = T(0);
      const T rc = T(2.5);

      for (int c = myCell - 1; c <= myCell + 1; c++) {

          if (c < 0 || c >= numCells) {
              continue;
          }

          for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {

              if (m == k) {
                  continue;
              }

              T dx = (T)(myR - sortedR[m]);
              T dist = dx < T(0) ? -dx : dx;

              if (dist < rc) {
                  T invR = T(1) / dist;
                  T sr = invR;
                  T sr6 = sr * sr * sr * sr * sr * sr;
                  T sr12 = sr6 * sr6;

                  T force =
                      T(24) * invR *
                      (T(2) * sr12 - sr6) - FC;

                  Fi += force * (dx * invR);
              }
          }
      }

      F_sorted[k] = Fi;
  }

  template <typename T>
  __global__ void initAccelKernel(double *a, const T *F, int N) {
      int i = blockIdx.x * blockDim.x + threadIdx.x;

      if (i >= N) {
          return;
      }

      a[i] = (double)F[i] / MASS;
  }

  template <typename T>
  __global__ void velocityKernel(
      double *v, double *a, const T *F, int N, double dt
  ) {
      int i = blockIdx.x * blockDim.x + threadIdx.x;

      if (i >= N) {
          return;
      }

      double a_new = (double)F[i] / MASS;

      v[i] += 0.5 * (a[i] + a_new) * dt;
      a[i] = a_new;
  }

  double computeKE(const std::vector<double>& v) {
      double KE = 0.0;

      for (int i = 0; i < N; i++) {
          KE += 0.5 * MASS * v[i] * v[i];
      }

      return KE;
  }

  double computePE(
      const std::vector<double>& r,
      const std::vector<int>& sortedIndex,
      const std::vector<int>& cellStart,
      double r_min,
      int numCells
  ) {
      double PE = 0.0;

      for (int i = 0; i < N; i++) {

          int myCell = (int)floor((r[i] - r_min) / LCELL);

          if (myCell < 0) {
              myCell = 0;
          }

          if (myCell >= numCells) {
              myCell = numCells - 1;
          }

          for (int c = myCell - 1; c <= myCell + 1; c++) {

              if (c < 0 || c >= numCells) {
                  continue;
              }

              for (int m = cellStart[c]; m < cellStart[c + 1]; m++) {

                  int j = sortedIndex[m];

                  if (j <= i) {
                      continue;
                  }

                  double dist = fabs(r[i] - r[j]);

                  if (dist < RC) {
                      PE += ljPotentialD(dist) - UC_D + FC_D * (dist - RC);
                  }
              }
          }
      }

      return PE;
  }

  template <typename T>
  void launchBuildAndForce(
      const double *d_r,
      int *d_cell,
      int *d_cellStart,
      int *d_flag,
      GridInfo *d_grid,
      T *d_F,
      int blocks,
      T FC
  ) {
      gridKernel<<<1, 1>>>(d_r, d_grid, d_flag, N);
      cellIndexKernel<<<blocks, TILE_SIZE>>>(d_r, d_cell, d_flag, d_grid, N);
      cellStartKernel<<<blocks, TILE_SIZE>>>(d_cell, d_cellStart, d_grid, N);
      cellListForceKernel<T><<<blocks, TILE_SIZE>>>(
          d_r, d_cellStart, d_F, d_grid, N, FC
      );
  }

  void readEnergy(
      const double *d_r,
      const double *d_v,
      const int *d_cellStart,
      std::vector<double>& r,
      std::vector<double>& v,
      std::vector<int>& cellStart,
      const std::vector<int>& sortedIndex,
      double& KE,
      double& PE
  ) {
      cudaMemcpy(r.data(), d_r, N * sizeof(double), cudaMemcpyDeviceToHost);
      cudaMemcpy(v.data(), d_v, N * sizeof(double), cudaMemcpyDeviceToHost);

      int numCells = (int)ceil((r[N - 1] - r[0]) / LCELL);
      if (numCells < 1) {
          numCells = 1;
      }

      cellStart.resize(numCells + 1);
      cudaMemcpy(cellStart.data(), d_cellStart, (numCells + 1) * sizeof(int),
                 cudaMemcpyDeviceToHost);

      KE = computeKE(v);
      PE = computePE(r, sortedIndex, cellStart, r[0], numCells);
  }

  template <typename T>
  int runSim(
      const char* tag,
      const std::vector<double>& r_init,
      const std::vector<double>& v_init,
      std::vector<double>& F0
  ) {
      double *d_r, *d_v, *d_a;
      T *d_F;
      int *d_cell, *d_cellStart, *d_flag;
      GridInfo *d_grid;

      cudaMalloc(&d_r, N * sizeof(double));
      cudaMalloc(&d_v, N * sizeof(double));
      cudaMalloc(&d_a, N * sizeof(double));
      cudaMalloc(&d_F, N * sizeof(T));
      cudaMalloc(&d_cell, N * sizeof(int));
      cudaMalloc(&d_cellStart, (N + 2) * sizeof(int));
      cudaMalloc(&d_flag, sizeof(int));
      cudaMalloc(&d_grid, sizeof(GridInfo));

      cudaMemcpy(d_r, r_init.data(), N * sizeof(double), cudaMemcpyHostToDevice);
      cudaMemcpy(d_v, v_init.data(), N * sizeof(double), cudaMemcpyHostToDevice);
      cudaMemset(d_flag, 0, sizeof(int));

      const int blocks = (N + TILE_SIZE - 1) / TILE_SIZE;
      const T FC = (T)FC_D;

      launchBuildAndForce<T>(d_r, d_cell, d_cellStart, d_flag, d_grid, d_F, blocks, FC);
      initAccelKernel<T><<<blocks, TILE_SIZE>>>(d_a, d_F, N);

      cudaError_t err = cudaDeviceSynchronize();
      if (err != cudaSuccess) {
          std::cerr << tag << ": CUDA error during init: "
                    << cudaGetErrorString(err) << '\n';
          exit(1);
      }

      std::vector<T> hF(N);
      cudaMemcpy(hF.data(), d_F, N * sizeof(T), cudaMemcpyDeviceToHost);
      F0.assign(N, 0.0);
      for (int i = 0; i < N; i++) {
          F0[i] = (double)hF[i];
      }

      std::vector<double> r(N), v(N);
      std::vector<int> cellStart;
      std::vector<int> sortedIndex(N);
      std::iota(sortedIndex.begin(), sortedIndex.end(), 0);

      char fname[128];
      snprintf(fname, sizeof(fname), "dump_traj_%s.csv", tag);
      FILE* ft = fopen(fname, "w");
      fprintf(ft, "step,KE,PE,E\n");

      double KE0 = 0.0;
      double PE0 = 0.0;
      readEnergy(d_r, d_v, d_cellStart, r, v, cellStart, sortedIndex, KE0, PE0);
      fprintf(ft, "0,%.17g,%.17g,%.17g\n", KE0, PE0, KE0 + PE0);

      const double E0 = KE0 + PE0;
      double lastRel = 0.0;

      for (int step = 1; step <= STEPS; step++) {

          positionKernel<<<blocks, TILE_SIZE>>>(d_r, d_v, d_a, N, DT);
          launchBuildAndForce<T>(d_r, d_cell, d_cellStart, d_flag, d_grid, d_F, blocks, FC);
          velocityKernel<T><<<blocks, TILE_SIZE>>>(d_v, d_a, d_F, N, DT);

          if (step % CHECKPOINT == 0) {
              err = cudaDeviceSynchronize();
              if (err != cudaSuccess) {
                  std::cerr << tag << ": CUDA error: "
                            << cudaGetErrorString(err) << '\n';
                  exit(1);
              }

              double KE = 0.0;
              double PE = 0.0;
              readEnergy(d_r, d_v, d_cellStart, r, v, cellStart, sortedIndex, KE, PE);

              double E = KE + PE;
              fprintf(ft, "%d,%.17g,%.17g,%.17g\n", step, KE, PE, E);

              lastRel = std::fabs(E - E0) / std::fabs(E0);
          }
      }

      fclose(ft);

      cudaDeviceSynchronize();

      int flag = 0;
      cudaMemcpy(&flag, d_flag, sizeof(int), cudaMemcpyDeviceToHost);

      cudaMemcpy(r.data(), d_r, N * sizeof(double), cudaMemcpyDeviceToHost);
      cudaMemcpy(v.data(), d_v, N * sizeof(double), cudaMemcpyDeviceToHost);

      snprintf(fname, sizeof(fname), "dump_final_%s.csv", tag);
      FILE* ff = fopen(fname, "w");
      fprintf(ff, "i,r,v\n");
      for (int i = 0; i < N; i++) {
          fprintf(ff, "%d,%.17g,%.17g\n", i, r[i], v[i]);
      }
      fclose(ff);

      std::cout << tag << ": KE0 = " << KE0 << ", PE0 = " << PE0
                << ", final |E-E0|/|E0| = " << lastRel
                << ", device flags = " << flag << "\n";

      cudaFree(d_r);
      cudaFree(d_v);
      cudaFree(d_a);
      cudaFree(d_F);
      cudaFree(d_cell);
    cudaFree(d_a);
    cudaFree(d_F);
    cudaFree(d_cell);
    cudaFree(d_cellStart);
    cudaFree(d_flag);
    cudaFree(d_grid);

    return flag;
}

int main() {

    std::vector<double> r_init(N), v_init(N);

    std::mt19937 gen(SEED);
    std::uniform_real_distribution<double> jitter(-0.2, 0.2);

    for (int i = 0; i < N; i++) {
        r_init[i] = i * 1.5 + jitter(gen);
        v_init[i] = (i % 2 == 0) ? 0.01 : -0.01;
    }

    std::cout << "N = " << N << ", seed = " << SEED << ", steps = " << STEPS
              << ", checkpoint = " << CHECKPOINT << "\n";

    std::vector<double> F_fp64;
    std::vector<double> F_mixed;

    int f1 = runSim<double>("fp64", r_init, v_init, F_fp64);
    int f2 = runSim<float>("mixed", r_init, v_init, F_mixed);

    FILE* fs = fopen("dump_state0.csv", "w");
    fprintf(fs, "i,r,v,F_fp64,F_mixed\n");
    for (int i = 0; i < N; i++) {
        fprintf(fs, "%d,%.17g,%.17g,%.17g,%.17g\n",
                i, r_init[i], v_init[i], F_fp64[i], F_mixed[i]);
    }
    fclose(fs);

    std::cout << "wrote dump_state0.csv, dump_traj_{fp64,mixed}.csv, "
                 "dump_final_{fp64,mixed}.csv\n";

    return (f1 | f2) ? 1 : 0;
}

In [16]:
!nvcc -O2 md_dump.cu -o md_dump && ./md_dump
!zip -q oracle_data.zip dump_*.csv && ls -la oracle_data.zip dump_*.csv

N = 4096, seed = 42, steps = 30000, checkpoint = 1000
fp64: KE0 = 0.2048, PE0 = -1308.88, final |E-E0|/|E0| = 3.00725e-08, device flags = 0
mixed: KE0 = 0.2048, PE0 = -1308.88, final |E-E0|/|E0| = 3.73858e-08, device flags = 0
wrote dump_state0.csv, dump_traj_{fp64,mixed}.csv, dump_final_{fp64,mixed}.csv
-rw-r--r-- 1 root root 180635 Sep 30 19:24 dump_final_fp64.csv
-rw-r--r-- 1 root root 180710 Sep 30 19:24 dump_final_mixed.csv
-rw-r--r-- 1 root root 283715 Sep 30 19:24 dump_state0.csv
-rw-r--r-- 1 root root   2008 Sep 30 19:24 dump_traj_fp64.csv
-rw-r--r-- 1 root root   2006 Sep 30 19:24 dump_traj_mixed.csv
-rw-r--r-- 1 root root 304556 Sep 30 19:24 oracle_data.zip


Results of the comparison (full output in `validation/results_n4096.txt`):

| Test | Result | Tolerance |
|---|---|---|
| FP64 forces at t = 0, maximum difference | 9.2e-15 | 1e-12 |
| Potential energy at t = 0, relative difference | 1.7e-16 | 1e-12 |
| FP64 energy over 30,000 steps, maximum difference relative to \|E0\| | 5.1e-13 | 1e-9 |
| Mixed forces at t = 0, maximum / rms difference | 5.1e-6 / 4.6e-7 | 2e-5 |
| FP64 final positions, maximum difference | 2.2e-10 | informational |
| Mixed final positions, maximum difference | 2.6e-6 | informational |

The FP64 code agrees with OpenMM to rounding error in forces, energy and trajectory, and OpenMM reproduces the 3e-8 energy error digit for digit (3.581e-8 at step 10,000 in both), which confirms it comes from the model and time step.

Against OpenMM, the mixed-precision energy deviates by up to 9.0e-9 of |E0| (at step 25,000; 7.3e-9 at step 30,000). It grows roughly linearly at first and then flattens, and 30,000 steps cannot distinguish a slow drift from a long-period oscillation. A longer run would be needed to say that mixed precision does not drift.

## 8. Summary

| Question | Answer |
|---|---|
| Can everything run in FP32? | No, with absolute coordinates: positions near 1e5 freeze (float spacing 0.0078, steps of 1e-6) and energy is created |
| Can the pair-force arithmetic run in FP32? | Yes: energy error 10 to 15% above the FP64 error (3.8e-8 to 3.9e-8 against 3.4e-8), force rms error 4.6e-7 |
| Speed | Force kernel 4.1x to 4.8x faster; 1.18x per step with host-driven cell lists, 2.06x with the state on the GPU |
| Correct? | FP64 code agrees with OpenMM to about 1e-14 in forces and 5e-13 in energy over 30,000 steps |

### Limits

- A 1D chain with about two neighbours per particle. Float accumulation errors will be larger with the 30 to 60 neighbours of a 3D liquid, so these energy results should not be extrapolated to 3D.
- One GPU, no profiler.
- The OpenMM check validates this implementation of the stated model, not the choice of model, and is done for N = 4,096 in 1D.
- The mixed-precision energy error against OpenMM was followed for 30,000 steps only.
- The GPU-resident cell list relies on particles never overtaking each other.
- Why the force kernel runs about twice as fast inside the GPU-resident loop as in the isolated benchmark has not been investigated.